# Final Open-Access Statistical Analysis

In [2]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import nolds
import spm1d

from scipy import stats
from scipy.interpolate import interp1d
from statsmodels.stats.multitest import multipletests
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side

OUTPUT_DIR = Path("Statistical_Output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ALPHA = 0.05
EXPECTED_REPETITIONS = 2


# =====================================================
# FORMATTING HELPERS
# =====================================================

def _fmt_p(p):
    if p is None or not np.isfinite(p):
        return "NA"
    if p < 0.001:
        return "< .001"
    return f"= {p:.3f}".replace("0.", ".")


def _mean_sd_ci(values, decimals=3):
    values = pd.Series(values, dtype=float).dropna()
    n = len(values)

    if n < 2:
        return "NA"

    mean_value = values.mean()
    sd_value = values.std(ddof=1)
    se_value = sd_value / np.sqrt(n)
    tcrit = stats.t.ppf(0.975, n - 1)

    ci_low = mean_value - tcrit * se_value
    ci_high = mean_value + tcrit * se_value

    return (
        f"{mean_value:.{decimals}f} ± {sd_value:.{decimals}f}\n"
        f"[{ci_low:.{decimals}f}, {ci_high:.{decimals}f}]"
    )


def _format_workbook(path):
    wb = load_workbook(path)

    thin = Side(style="thin", color="B7B7B7")
    fill = PatternFill(fill_type="solid", fgColor="D9EAF7")

    for ws in wb.worksheets:
        ws.freeze_panes = "A2"

        for cell in ws[1]:
            cell.font = Font(bold=True)
            cell.fill = fill
            cell.alignment = Alignment(
                horizontal="center",
                vertical="center",
                wrap_text=True,
            )
            cell.border = Border(
                left=thin,
                right=thin,
                top=thin,
                bottom=thin,
            )

        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(
                    vertical="center",
                    wrap_text=True,
                )
                cell.border = Border(
                    left=thin,
                    right=thin,
                    top=thin,
                    bottom=thin,
                )

        for col in ws.columns:
            letter = col[0].column_letter
            max_len = min(
                60,
                max(
                    10,
                    max(
                        len(str(c.value))
                        if c.value is not None
                        else 0
                        for c in col
                    ) + 2,
                ),
            )
            ws.column_dimensions[letter].width = max_len

    wb.save(path)


# =====================================================
# PARTICIPANT-LEVEL DATA
# =====================================================

def _participant_condition_means(
    long_data,
    metric_col,
    value_col,
    subject_col,
    condition_col,
    condition_order,
    expected_repetitions=2,
):
    """
    Convert repeated observations to ONE participant-level value per condition.

    Missing-repetition rule used throughout this notebook:
      - 2 valid repetitions -> average the 2
      - 1 valid repetition  -> use the 1 available repetition
      - 0 valid repetitions -> no participant-condition value

    A participant is excluded from a specific inferential test only when
    an entire condition required for that test has no valid value.
    Repetitions are never treated as independent inferential observations.
    """

    required = [metric_col, value_col, subject_col, condition_col]
    missing = [c for c in required if c not in long_data.columns]
    if missing:
        raise ValueError(
            f"Missing columns for repeated-measures analysis: {missing}"
        )

    d = long_data[required].copy()
    d.columns = ["_Metric", "_Y", "_Subject", "_Condition"]
    d["_Y"] = pd.to_numeric(d["_Y"], errors="coerce")

    d = d.dropna(
        subset=["_Metric", "_Y", "_Subject", "_Condition"]
    )
    d = d[d["_Condition"].isin(condition_order)].copy()

    d["_Condition"] = pd.Categorical(
        d["_Condition"],
        categories=condition_order,
        ordered=True,
    )

    # sort=False preserves the metric order supplied by each analysis cell.
    participant_means = (
        d.groupby(
            ["_Metric", "_Subject", "_Condition"],
            observed=True,
            as_index=False,
            sort=False,
        )
        .agg(
            Valid_Trials=("_Y", "count"),
            Value=("_Y", "mean"),
        )
    )

    participant_means["Expected_Repetitions"] = expected_repetitions
    participant_means["Repetition_Status"] = np.select(
        [
            participant_means["Valid_Trials"] >= expected_repetitions,
            participant_means["Valid_Trials"] == 1,
        ],
        [
            "All expected repetitions available",
            "One valid repetition used",
        ],
        default="Available repetitions averaged",
    )

    # IMPORTANT: do NOT set Value to NaN when only one repetition exists.
    # The mean of the available repetition(s) is retained.
    return d, participant_means


def _complete_wide(
    participant_means_metric,
    condition_order,
):
    """
    Return only participants with a complete value in every condition.
    """

    wide = (
        participant_means_metric
        .pivot(
            index="_Subject",
            columns="_Condition",
            values="Value",
        )
        .reindex(columns=condition_order)
        .dropna()
    )

    return wide


# =====================================================
# EFFECT SIZE HELPERS
# =====================================================

def _cohen_dz(differences):
    """
    Cohen's dz for paired differences.

    Positive dz means the second condition is larger than the first
    condition when differences are defined as second - first.
    """

    differences = np.asarray(
        differences,
        dtype=float,
    )

    differences = differences[
        np.isfinite(differences)
    ]

    if len(differences) < 2:
        return np.nan

    sd_difference = np.std(
        differences,
        ddof=1,
    )

    if not np.isfinite(sd_difference) or sd_difference == 0:
        return np.nan

    return float(
        np.mean(differences)
        / sd_difference
    )


def _paired_difference_summary(
    first,
    second,
):
    """
    Summary for second - first.
    """

    first = np.asarray(first, dtype=float)
    second = np.asarray(second, dtype=float)

    difference = second - first
    n = len(difference)

    if n < 2:
        return {
            "Mean Difference": np.nan,
            "SE": np.nan,
            "95% CI Low": np.nan,
            "95% CI High": np.nan,
            "Cohen dz": np.nan,
        }

    mean_difference = np.mean(difference)
    sd_difference = np.std(
        difference,
        ddof=1,
    )
    se_difference = sd_difference / np.sqrt(n)
    tcrit = stats.t.ppf(
        0.975,
        n - 1,
    )

    return {
        "Mean Difference": float(mean_difference),
        "SE": float(se_difference),
        "95% CI Low": float(
            mean_difference
            - tcrit * se_difference
        ),
        "95% CI High": float(
            mean_difference
            + tcrit * se_difference
        ),
        "Cohen dz": _cohen_dz(difference),
    }


# =====================================================
# TWO-CONDITION PRIMARY TEST: PAIRED t-TEST
# =====================================================

def _paired_t_test(
    wide,
    condition_order,
):
    first_name = condition_order[0]
    second_name = condition_order[1]

    first = wide[first_name].to_numpy(float)
    second = wide[second_name].to_numpy(float)

    n = len(wide)

    if n < 2:
        return {
            "t": np.nan,
            "df": np.nan,
            "p": np.nan,
            "N": n,
            "Shapiro-Wilk W": np.nan,
            "Shapiro-Wilk p": np.nan,
            **_paired_difference_summary(
                first,
                second,
            ),
        }

    difference = second - first

    t_statistic, p_value = stats.ttest_rel(
        second,
        first,
        nan_policy="omit",
    )

    if 3 <= n <= 5000:
        shapiro_w, shapiro_p = stats.shapiro(
            difference
        )
    else:
        shapiro_w, shapiro_p = np.nan, np.nan

    return {
        "t": float(t_statistic),
        "df": int(n - 1),
        "p": float(p_value),
        "N": int(n),
        "Shapiro-Wilk W": (
            float(shapiro_w)
            if np.isfinite(shapiro_w)
            else np.nan
        ),
        "Shapiro-Wilk p": (
            float(shapiro_p)
            if np.isfinite(shapiro_p)
            else np.nan
        ),
        **_paired_difference_summary(
            first,
            second,
        ),
    }


# =====================================================
# SPHERICITY
# =====================================================

def _greenhouse_geisser_epsilon(y):
    """
    Greenhouse-Geisser epsilon for a one-factor repeated-measures design.

    y shape = participants x conditions.
    """

    y = np.asarray(y, dtype=float)
    n, k = y.shape

    if k <= 2:
        return 1.0

    covariance = np.cov(
        y,
        rowvar=False,
        ddof=1,
    )

    mean_diag = np.mean(
        np.diag(covariance)
    )
    grand_mean_cov = np.mean(
        covariance
    )
    row_means = np.mean(
        covariance,
        axis=1,
    )

    numerator = (
        k**2
        * (
            mean_diag
            - grand_mean_cov
        )**2
    )

    denominator = (
        (k - 1)
        * (
            np.sum(covariance**2)
            - 2 * k * np.sum(row_means**2)
            + k**2 * grand_mean_cov**2
        )
    )

    if (
        not np.isfinite(denominator)
        or denominator <= 0
    ):
        return 1.0

    epsilon = numerator / denominator

    lower_bound = 1.0 / (k - 1)

    return float(
        np.clip(
            epsilon,
            lower_bound,
            1.0,
        )
    )


def _mauchly_sphericity(y):
    """
    Mauchly's test of sphericity for one within-subject factor.

    Returns W, chi-square, df, and p.

    With only two conditions, sphericity is automatically satisfied.
    """

    y = np.asarray(y, dtype=float)
    n, k = y.shape

    if k <= 2:
        return {
            "Mauchly W": np.nan,
            "Mauchly chi-square": np.nan,
            "Mauchly df": 1,
            "Mauchly p": 1.0,
            "Sphericity Met": True,
        }

    covariance = np.cov(
        y,
        rowvar=False,
        ddof=1,
    )

    centering = (
        np.eye(k)
        - np.ones((k, k)) / k
    )

    double_centered = (
        centering
        @ covariance
        @ centering
    )

    eigenvalues = np.linalg.eigvalsh(
        double_centered
    )

    tolerance = (
        np.max(np.abs(eigenvalues))
        * 1e-10
        if np.max(np.abs(eigenvalues)) > 0
        else 1e-12
    )

    eigenvalues = eigenvalues[
        eigenvalues > tolerance
    ]

    # We need exactly k - 1 non-zero eigenvalues.
    if len(eigenvalues) != (k - 1):
        return {
            "Mauchly W": np.nan,
            "Mauchly chi-square": np.nan,
            "Mauchly df": int(
                k * (k - 1) / 2 - 1
            ),
            "Mauchly p": np.nan,
            "Sphericity Met": False,
        }

    mean_eigenvalue = np.mean(
        eigenvalues
    )

    if (
        mean_eigenvalue <= 0
        or np.any(eigenvalues <= 0)
    ):
        return {
            "Mauchly W": np.nan,
            "Mauchly chi-square": np.nan,
            "Mauchly df": int(
                k * (k - 1) / 2 - 1
            ),
            "Mauchly p": np.nan,
            "Sphericity Met": False,
        }

    W = (
        np.prod(eigenvalues)
        / (
            mean_eigenvalue
            ** (k - 1)
        )
    )

    W = float(
        np.clip(
            W,
            np.finfo(float).tiny,
            1.0,
        )
    )

    f_correction = (
        (
            2 * (k - 1)**2
            + k
            + 1
        )
        / (
            6
            * (k - 1)
            * (n - 1)
        )
    )

    chi_square = (
        (f_correction - 1)
        * (n - 1)
        * np.log(W)
    )

    df = int(
        k * (k - 1) / 2
        - 1
    )

    p_value = stats.chi2.sf(
        chi_square,
        df,
    )

    return {
        "Mauchly W": W,
        "Mauchly chi-square": float(
            chi_square
        ),
        "Mauchly df": df,
        "Mauchly p": float(
            p_value
        ),
        "Sphericity Met": bool(
            p_value >= ALPHA
        ),
    }


# =====================================================
# THREE-LEVEL PRIMARY TEST: REPEATED-MEASURES ANOVA
# =====================================================

def _rm_anova_oneway(
    wide,
    condition_order,
):
    """
    One-way repeated-measures ANOVA using complete participant-level
    condition means.

    If Mauchly's test indicates a sphericity violation (p < .05),
    Greenhouse-Geisser corrected degrees of freedom and p-value are
    used for primary inference.
    """

    y = wide[
        condition_order
    ].to_numpy(float)

    n, k = y.shape

    if n < 2 or k < 3:
        return {
            "F": np.nan,
            "df1 uncorrected": np.nan,
            "df2 uncorrected": np.nan,
            "df1 reported": np.nan,
            "df2 reported": np.nan,
            "p uncorrected": np.nan,
            "p GG corrected": np.nan,
            "p": np.nan,
            "Partial Eta Squared": np.nan,
            "GG epsilon": np.nan,
            "Correction Used": "NA",
            "N": n,
            "Shapiro-Wilk W": np.nan,
            "Shapiro-Wilk p": np.nan,
            **_mauchly_sphericity(y),
        }

    grand_mean = np.mean(y)
    condition_means = np.mean(
        y,
        axis=0,
    )
    subject_means = np.mean(
        y,
        axis=1,
    )

    ss_total = np.sum(
        (y - grand_mean)**2
    )

    ss_condition = (
        n
        * np.sum(
            (
                condition_means
                - grand_mean
            )**2
        )
    )

    ss_subject = (
        k
        * np.sum(
            (
                subject_means
                - grand_mean
            )**2
        )
    )

    ss_error = (
        ss_total
        - ss_condition
        - ss_subject
    )

    # Protect against tiny negative round-off.
    if ss_error < 0 and abs(ss_error) < 1e-12:
        ss_error = 0.0

    df1 = k - 1
    df2 = (
        (n - 1)
        * (k - 1)
    )

    ms_condition = (
        ss_condition
        / df1
    )

    ms_error = (
        ss_error
        / df2
    )

    F_value = (
        ms_condition
        / ms_error
        if ms_error > 0
        else np.nan
    )

    p_uncorrected = (
        stats.f.sf(
            F_value,
            df1,
            df2,
        )
        if np.isfinite(F_value)
        else np.nan
    )

    eta_partial = (
        ss_condition
        / (
            ss_condition
            + ss_error
        )
        if (
            ss_condition
            + ss_error
        ) > 0
        else np.nan
    )

    epsilon_gg = (
        _greenhouse_geisser_epsilon(
            y
        )
    )

    df1_gg = (
        epsilon_gg
        * df1
    )

    df2_gg = (
        epsilon_gg
        * df2
    )

    p_gg = (
        stats.f.sf(
            F_value,
            df1_gg,
            df2_gg,
        )
        if np.isfinite(F_value)
        else np.nan
    )

    sphericity = _mauchly_sphericity(
        y
    )

    # Residuals for a one-factor repeated-measures ANOVA:
    # observation - participant mean - condition mean + grand mean
    residuals = (
        y
        - subject_means[:, None]
        - condition_means[None, :]
        + grand_mean
    ).ravel()

    if 3 <= len(residuals) <= 5000:
        shapiro_w, shapiro_p = stats.shapiro(
            residuals
        )
    else:
        shapiro_w, shapiro_p = np.nan, np.nan

    # Greenhouse-Geisser correction is applied only when
    # Mauchly's test indicates violated sphericity.
    if (
        np.isfinite(
            sphericity["Mauchly p"]
        )
        and sphericity["Mauchly p"] < ALPHA
    ):
        reported_p = p_gg
        reported_df1 = df1_gg
        reported_df2 = df2_gg
        correction_used = (
            "Greenhouse-Geisser"
        )
    else:
        reported_p = p_uncorrected
        reported_df1 = float(df1)
        reported_df2 = float(df2)
        correction_used = "None"

    return {
        "F": float(F_value),
        "df1 uncorrected": int(df1),
        "df2 uncorrected": int(df2),
        "df1 reported": float(
            reported_df1
        ),
        "df2 reported": float(
            reported_df2
        ),
        "p uncorrected": float(
            p_uncorrected
        ),
        "p GG corrected": float(
            p_gg
        ),
        "p": float(
            reported_p
        ),
        "Partial Eta Squared": float(
            eta_partial
        ),
        "GG epsilon": float(
            epsilon_gg
        ),
        "Correction Used": (
            correction_used
        ),
        "N": int(n),
        "Shapiro-Wilk W": (
            float(shapiro_w)
            if np.isfinite(shapiro_w)
            else np.nan
        ),
        "Shapiro-Wilk p": (
            float(shapiro_p)
            if np.isfinite(shapiro_p)
            else np.nan
        ),
        **sphericity,
    }


# =====================================================
# NONPARAMETRIC SENSITIVITY ANALYSIS
# =====================================================

def _sensitivity_test(
    wide,
    condition_order,
):
    n = len(wide)

    if n < 3:
        return {
            "Sensitivity Test":
                "Insufficient complete participants",
            "Sensitivity Statistic": np.nan,
            "Sensitivity df": np.nan,
            "Sensitivity p": np.nan,
            "Sensitivity N": int(n),
        }

    if len(condition_order) == 2:
        first = wide[
            condition_order[0]
        ].to_numpy(float)

        second = wide[
            condition_order[1]
        ].to_numpy(float)

        difference = second - first

        if np.allclose(
            difference,
            0,
        ):
            statistic = 0.0
            p_value = 1.0
        else:
            try:
                statistic, p_value = stats.wilcoxon(
                    second,
                    first,
                    alternative="two-sided",
                    zero_method="wilcox",
                )
            except ValueError:
                statistic, p_value = (
                    np.nan,
                    np.nan,
                )

        return {
            "Sensitivity Test":
                "Wilcoxon signed-rank",
            "Sensitivity Statistic":
                float(statistic),
            "Sensitivity df": np.nan,
            "Sensitivity p":
                float(p_value),
            "Sensitivity N": int(n),
        }

    arrays = [
        wide[level].to_numpy(float)
        for level in condition_order
    ]

    statistic, p_value = (
        stats.friedmanchisquare(
            *arrays
        )
    )

    return {
        "Sensitivity Test":
            "Friedman",
        "Sensitivity Statistic":
            float(statistic),
        "Sensitivity df":
            int(
                len(condition_order)
                - 1
            ),
        "Sensitivity p":
            float(p_value),
        "Sensitivity N":
            int(n),
    }


# =====================================================
# PAIRWISE POST-HOC PAIRED t-TESTS
# =====================================================

def _paired_posthoc(
    wide,
    condition_order,
):
    """
    All pairwise paired-samples t-tests.
    Holm adjustment is applied within the metric.
    """

    rows = []

    for i in range(
        len(condition_order) - 1
    ):
        for j in range(
            i + 1,
            len(condition_order)
        ):
            first_name = (
                condition_order[i]
            )
            second_name = (
                condition_order[j]
            )

            first = wide[
                first_name
            ].to_numpy(float)

            second = wide[
                second_name
            ].to_numpy(float)

            n = len(wide)

            t_statistic, p_value = (
                stats.ttest_rel(
                    second,
                    first,
                    nan_policy="omit",
                )
            )

            summary = (
                _paired_difference_summary(
                    first,
                    second,
                )
            )

            rows.append({
                "Comparison":
                    f"{second_name} − {first_name}",
                "N": int(n),
                **summary,
                "t": float(
                    t_statistic
                ),
                "df": int(
                    n - 1
                ),
                "Raw p": float(
                    p_value
                ),
            })

    posthoc = pd.DataFrame(rows)

    if posthoc.empty:
        return posthoc

    valid = (
        posthoc["Raw p"].notna()
        & np.isfinite(
            posthoc[
                "Raw p"
            ].to_numpy(float)
        )
    )

    posthoc[
        "Holm-adjusted p"
    ] = np.nan

    posthoc[
        "Significant After Holm"
    ] = False

    if valid.any():
        reject, adjusted_p, _, _ = (
            multipletests(
                posthoc.loc[
                    valid,
                    "Raw p",
                ],
                alpha=ALPHA,
                method="holm",
            )
        )

        posthoc.loc[
            valid,
            "Holm-adjusted p",
        ] = adjusted_p

        posthoc.loc[
            valid,
            "Significant After Holm",
        ] = reject

    return posthoc


# =====================================================
# MAIN DISPATCHER
# =====================================================

def run_repeated_family(
    long_data,
    metric_col,
    value_col,
    subject_col,
    condition_col,
    condition_order,
    output_filename,
    expected_repetitions=EXPECTED_REPETITIONS,
    decimals=3,
):

    # --------------------------------------------------------
    # Small effect-size helpers
    # --------------------------------------------------------

    def rank_biserial(diff):
        diff = np.asarray(diff, float)
        diff = diff[np.isfinite(diff) & (diff != 0)]

        if len(diff) == 0:
            return np.nan

        ranks = stats.rankdata(np.abs(diff))

        pos = ranks[diff > 0].sum()
        neg = ranks[diff < 0].sum()

        return (pos - neg) / (pos + neg)


    def friedman_w(statistic, n, k):
        if n == 0 or k <= 1:
            return np.nan

        return statistic / (n * (k - 1))


    # --------------------------------------------------------
    # Participant × condition means
    # --------------------------------------------------------

    raw_data, participant_means = (
        _participant_condition_means(
            long_data=long_data,
            metric_col=metric_col,
            value_col=value_col,
            subject_col=subject_col,
            condition_col=condition_col,
            condition_order=condition_order,
            expected_repetitions=expected_repetitions,
        )
    )

    result_rows = []
    diagnostic_rows = []
    wide_by_metric = {}

    # ========================================================
    # PRIMARY ANALYSES
    # ========================================================

    for metric in pd.unique(
        participant_means["_Metric"]
    ):

        pm = participant_means[
            participant_means["_Metric"] == metric
        ].copy()

        wide = _complete_wide(
            pm,
            condition_order,
        )

        wide_by_metric[metric] = wide.copy()

        n = len(wide)

        if n < 2:
            continue

        descriptives = {
            condition:
                _mean_sd_ci(
                    wide[condition],
                    decimals=decimals,
                )
            for condition in condition_order
        }

        # ====================================================
        # TWO CONDITIONS
        # ====================================================

        if len(condition_order) == 2:

            c1, c2 = condition_order

            first = wide[c1].to_numpy(float)
            second = wide[c2].to_numpy(float)

            diff = second - first

            # Normality of paired differences
            if 3 <= n <= 5000:
                sw, sp = stats.shapiro(diff)
            else:
                sw, sp = np.nan, np.nan

            normal = (
                np.isfinite(sp)
                and sp >= ALPHA
            )

            # -----------------------------------------------
            # NORMAL -> paired t-test
            # -----------------------------------------------

            if normal:

                test = _paired_t_test(
                    wide,
                    condition_order,
                )

                result_rows.append({
                    "Metric": metric,
                    "Participants": n,
                    **descriptives,

                    "Primary Test":
                        "Paired-samples t-test",

                    "Contrast":
                        f"{c2} − {c1}",

                    "Statistic":
                        test["t"],

                    "df":
                        test["df"],

                    "Effect Size":
                        test["Cohen dz"],

                    "Effect Size Type":
                        "Cohen dz",

                    "Raw p":
                        test["p"],
                })

            # -----------------------------------------------
            # NON-NORMAL -> Wilcoxon
            # -----------------------------------------------

            else:

                if np.allclose(diff, 0):
                    statistic = 0.0
                    p_value = 1.0

                else:
                    statistic, p_value = (
                        stats.wilcoxon(
                            second,
                            first,
                            alternative="two-sided",
                            zero_method="wilcox",
                        )
                    )

                result_rows.append({
                    "Metric": metric,
                    "Participants": n,
                    **descriptives,

                    "Primary Test":
                        "Wilcoxon signed-rank",

                    "Contrast":
                        f"{c2} − {c1}",

                    "Statistic":
                        statistic,

                    "df":
                        np.nan,

                    "Effect Size":
                        rank_biserial(diff),

                    "Effect Size Type":
                        "Rank-biserial r",

                    "Raw p":
                        p_value,
                })

            diagnostic_rows.append({
                "Metric": metric,

                "Normality Object":
                    f"{c2} − {c1} paired differences",

                "Shapiro-Wilk W":
                    sw,

                "Shapiro-Wilk p":
                    sp,

                "Normality Met":
                    normal,

                "Selected Test":
                    (
                        "Paired-samples t-test"
                        if normal
                        else
                        "Wilcoxon signed-rank"
                    ),

                "Mauchly W":
                    np.nan,

                "Mauchly p":
                    np.nan,

                "GG epsilon":
                    np.nan,

                "Correction Used":
                    "Not applicable",
            })


        # ====================================================
        # THREE OR MORE CONDITIONS
        # ====================================================

        else:

            # Existing function already calculates:
            # RM residual Shapiro
            # RM-ANOVA
            # Mauchly
            # GG correction
            anova = _rm_anova_oneway(
                wide,
                condition_order,
            )

            sp = anova[
                "Shapiro-Wilk p"
            ]

            normal = (
                np.isfinite(sp)
                and sp >= ALPHA
            )

            # -----------------------------------------------
            # NORMAL -> RM-ANOVA
            # -----------------------------------------------

            if normal:

                result_rows.append({
                    "Metric": metric,
                    "Participants": n,
                    **descriptives,

                    "Primary Test":
                        "Repeated-measures ANOVA",

                    "Statistic":
                        anova["F"],

                    "df1":
                        anova["df1 reported"],

                    "df2":
                        anova["df2 reported"],

                    "Effect Size":
                        anova[
                            "Partial Eta Squared"
                        ],

                    "Effect Size Type":
                        "Partial Eta Squared",

                    "Mauchly W":
                        anova["Mauchly W"],

                    "Mauchly p":
                        anova["Mauchly p"],

                    "GG epsilon":
                        anova["GG epsilon"],

                    "Correction Used":
                        anova["Correction Used"],

                    "Raw p":
                        anova["p"],
                })

            # -----------------------------------------------
            # NON-NORMAL -> Friedman
            # -----------------------------------------------

            else:

                arrays = [
                    wide[c].to_numpy(float)
                    for c in condition_order
                ]

                statistic, p_value = (
                    stats.friedmanchisquare(
                        *arrays
                    )
                )

                result_rows.append({
                    "Metric": metric,
                    "Participants": n,
                    **descriptives,

                    "Primary Test":
                        "Friedman test",

                    "Statistic":
                        statistic,

                    "df1":
                        len(condition_order) - 1,

                    "df2":
                        np.nan,

                    "Effect Size":
                        friedman_w(
                            statistic,
                            n,
                            len(condition_order),
                        ),

                    "Effect Size Type":
                        "Kendall W",

                    "Mauchly W":
                        np.nan,

                    "Mauchly p":
                        np.nan,

                    "GG epsilon":
                        np.nan,

                    "Correction Used":
                        "Not applicable",

                    "Raw p":
                        p_value,
                })

            diagnostic_rows.append({
                "Metric":
                    metric,

                "Normality Object":
                    "RM-ANOVA residuals",

                "Shapiro-Wilk W":
                    anova[
                        "Shapiro-Wilk W"
                    ],

                "Shapiro-Wilk p":
                    sp,

                "Normality Met":
                    normal,

                "Selected Test":
                    (
                        "Repeated-measures ANOVA"
                        if normal
                        else
                        "Friedman test"
                    ),

                "Mauchly W":
                    (
                        anova["Mauchly W"]
                        if normal
                        else np.nan
                    ),

                "Mauchly p":
                    (
                        anova["Mauchly p"]
                        if normal
                        else np.nan
                    ),

                "GG epsilon":
                    (
                        anova["GG epsilon"]
                        if normal
                        else np.nan
                    ),

                "Correction Used":
                    (
                        anova["Correction Used"]
                        if normal
                        else
                        "Not applicable"
                    ),
            })


    # ========================================================
    # RESULTS
    # ========================================================

    results = pd.DataFrame(
        result_rows
    )

    diagnostics = pd.DataFrame(
        diagnostic_rows
    )

    if results.empty:
        raise ValueError(
            "No usable participant-level data."
        )


    # ========================================================
    # FDR — ONLY ON SELECTED PRIMARY TEST
    # ========================================================

    results["FDR q"] = np.nan
    results["Significant After FDR"] = False

    valid = results["Raw p"].notna()

    if valid.any():

        reject, q, _, _ = multipletests(
            results.loc[
                valid,
                "Raw p",
            ],
            alpha=ALPHA,
            method="fdr_bh",
        )

        results.loc[
            valid,
            "FDR q",
        ] = q

        results.loc[
            valid,
            "Significant After FDR",
        ] = reject


    # ========================================================
    # MANUSCRIPT-READY SIGNIFICANCE TEXT
    # ========================================================

    def significance_text(row):

        p = _fmt_p(
            row["Raw p"]
        )

        q = _fmt_p(
            row["FDR q"]
        )

        test = row[
            "Primary Test"
        ]

        if test == "Paired-samples t-test":

            return (
                f"t({int(row['df'])}) = "
                f"{row['Statistic']:.3f}; "
                f"p {p}; q {q}; "
                f"dz = {row['Effect Size']:.3f}"
            )

        if test == "Wilcoxon signed-rank":

            return (
                f"W = {row['Statistic']:.3f}; "
                f"p {p}; q {q}; "
                f"r_rb = {row['Effect Size']:.3f}"
            )

        if test == "Repeated-measures ANOVA":

            correction = (
                "GG-corrected; "
                if row["Correction Used"]
                == "Greenhouse-Geisser"
                else ""
            )

            return (
                f"F({row['df1']:.3f}, "
                f"{row['df2']:.3f}) = "
                f"{row['Statistic']:.3f}; "
                f"{correction}"
                f"p {p}; q {q}; "
                f"ηp² = {row['Effect Size']:.3f}"
            )

        if test == "Friedman test":

            return (
                f"χ²({int(row['df1'])}) = "
                f"{row['Statistic']:.3f}; "
                f"p {p}; q {q}; "
                f"Kendall's W = "
                f"{row['Effect Size']:.3f}"
            )

        return "NA"


    results["Significance"] = (
        results.apply(
            significance_text,
            axis=1,
        )
    )


    # ========================================================
    # PAPER TABLE
    # ========================================================

    paper_columns = (
        [
            "Metric",
            "Participants",
        ]
        + list(condition_order)
        + [
            "Primary Test",
            "Significance",
        ]
    )

    paper_table = results[
        paper_columns
    ].copy()


    # ========================================================
    # POST HOC — ONLY FOR SIGNIFICANT 3-CONDITION OUTCOMES
    #
    # ANOVA -> paired t-tests
    # Friedman -> Wilcoxon
    # Holm adjustment within metric
    # ========================================================

    posthoc_rows = []

    if len(condition_order) > 2:

        significant_metrics = (
            results.loc[
                results[
                    "Significant After FDR"
                ],
                "Metric",
            ].tolist()
        )

        for metric in significant_metrics:

            wide = wide_by_metric[
                metric
            ]

            primary_test = results.loc[
                results["Metric"] == metric,
                "Primary Test",
            ].iloc[0]

            pair_rows = []

            for i in range(
                len(condition_order) - 1
            ):

                for j in range(
                    i + 1,
                    len(condition_order)
                ):

                    c1 = condition_order[i]
                    c2 = condition_order[j]

                    first = wide[
                        c1
                    ].to_numpy(float)

                    second = wide[
                        c2
                    ].to_numpy(float)

                    diff = second - first

                    # ANOVA -> paired t
                    if (
                        primary_test
                        == "Repeated-measures ANOVA"
                    ):

                        stat, p_value = (
                            stats.ttest_rel(
                                second,
                                first,
                            )
                        )

                        test_name = (
                            "Paired-samples t-test"
                        )

                        effect = _cohen_dz(
                            diff
                        )

                        effect_name = (
                            "Cohen dz"
                        )

                    # Friedman -> Wilcoxon
                    else:

                        stat, p_value = (
                            stats.wilcoxon(
                                second,
                                first,
                                alternative="two-sided",
                            )
                        )

                        test_name = (
                            "Wilcoxon signed-rank"
                        )

                        effect = (
                            rank_biserial(
                                diff
                            )
                        )

                        effect_name = (
                            "Rank-biserial r"
                        )

                    pair_rows.append({
                        "Metric":
                            metric,

                        "Comparison":
                            f"{c2} − {c1}",

                        "N":
                            len(wide),

                        "Test":
                            test_name,

                        "Statistic":
                            stat,

                        "Raw p":
                            p_value,

                        "Effect Size":
                            effect,

                        "Effect Size Type":
                            effect_name,
                    })

            pair_df = pd.DataFrame(
                pair_rows
            )

            if not pair_df.empty:

                _, adjusted, _, _ = (
                    multipletests(
                        pair_df["Raw p"],
                        alpha=ALPHA,
                        method="holm",
                    )
                )

                pair_df[
                    "Holm-adjusted p"
                ] = adjusted

                posthoc_rows.append(
                    pair_df
                )

    posthoc = (
        pd.concat(
            posthoc_rows,
            ignore_index=True,
        )
        if posthoc_rows
        else pd.DataFrame()
    )


    # ========================================================
    # METADATA
    # ========================================================

    metadata = pd.DataFrame({
        "Item": [
            "Experimental unit",
            "Repeated-trial handling",
            "Normality rule",
            "Two-condition test",
            "Three-condition test",
            "Sphericity",
            "Multiplicity",
            "Sensitivity analysis",
        ],

        "Specification": [
            "Participant",

            (
                "Available repetitions averaged within "
                "participant and condition; one valid "
                "repetition retained when necessary."
            ),

            (
                "Shapiro-Wilk on paired differences for "
                "two-condition analyses and RM residuals "
                "for three-condition analyses."
            ),

            (
                "Paired t-test when normal; "
                "Wilcoxon signed-rank when non-normal."
            ),

            (
                "RM-ANOVA when normal; "
                "Friedman when non-normal."
            ),

            (
                "Mauchly test and Greenhouse-Geisser "
                "correction apply only to RM-ANOVA."
            ),

            (
                "Benjamini-Hochberg FDR applied to the "
                "selected primary tests."
            ),

            "None.",
        ],
    })


    # ========================================================
    # SAVE
    # ========================================================

    output_path = (
        OUTPUT_DIR
        / output_filename
    )

    with pd.ExcelWriter(
        output_path,
        engine="openpyxl",
    ) as writer:

        paper_table.to_excel(
            writer,
            sheet_name="Paper Table",
            index=False,
        )

        results.to_excel(
            writer,
            sheet_name="Full Statistical Output",
            index=False,
        )

        diagnostics.to_excel(
            writer,
            sheet_name="Assumption Checks",
            index=False,
        )

        if not posthoc.empty:
            posthoc.to_excel(
                writer,
                sheet_name="Post Hoc Tests",
                index=False,
            )

        participant_means.to_excel(
            writer,
            sheet_name="Participant Means",
            index=False,
        )

        raw_data.to_excel(
            writer,
            sheet_name="Trial-Level Input Data",
            index=False,
        )

        metadata.to_excel(
            writer,
            sheet_name="Analysis Metadata",
            index=False,
        )

    _format_workbook(
        output_path
    )

    print(
        f"\nSaved: {output_path}"
    )

    display(
        paper_table
    )

    if not posthoc.empty:
        print(
            "\nHolm-adjusted post-hoc comparisons:"
        )
        display(
            posthoc
        )

    # Keep 5 outputs so your downstream notebook
    # structure does not break.
    sensitivity = pd.DataFrame()

    return (
        paper_table,
        results,
        diagnostics,
        sensitivity,
        posthoc,
    )

## Table: Sample Entropy

In [4]:
# =====================================================
# PARAMETERS
# =====================================================

fs = 60  # Sampling frequency, Hz

participants = [
    f"P{i:03d}"
    for i in range(1, 17)
]

conditions = {
    "LOW": [3, 4],
    "HIGH": [5, 6],
}

# Sample entropy parameters
EMBEDDING_DIMENSION = 2
TOLERANCE_RATIO = 0.20
MIN_SIGNAL_LENGTH = 50


# =====================================================
# TIMING DICTIONARIES, SECONDS
# =====================================================

cut_times = {
    "P001": [12.55, 15.17, 6.90, 7.56],
    "P002": [15.81, 15.86, 7.40, 6.00],
}

reaction_times = {
    "P003": {
        "start": [10.28, 13.53, 7.50, 8.15],
        "end": [21.50, 23.80, 13.50, 12.50],
    },
    "P004": {
        "start": [8.31, 12.50, 5.88, 5.18],
        "end": [18.00, 22.00, 11.40, 9.80],
    },
    "P005": {
        "start": [12.16, 11.78, 5.58, 6.03],
        "end": [20.60, 20.80, 10.00, 10.50],
    },
    "P006": {
        "start": [10.66, 12.90, 5.81, 6.25],
        "end": [19.70, 20.50, 7.20, 9.80],
    },
    "P007": {
        "start": [10.00, 12.00, 6.10, 6.00],
        "end": [23.70, 15.60, 8.00, 9.60],
    },
    "P008": {
        "start": [11.00, 12.00, 6.00, 7.00],
        "end": [19.60, 24.00, 11.80, 10.20],
    },
    "P009": {
        "start": [10.29, 10.56, 5.86, 6.02],
        "end": [20.50, 23.00, 9.00, 12.00],
    },
    "P010": {
        "start": [12.00, 12.00, 6.00, 16.50],
        "end": [15.50, 16.20, 11.00, 20.00],
    },
    "P011": {
        "start": [10.94, 12.76, 6.10, 7.91],
        "end": [20.80, 24.00, 9.30, 14.00],
    },
    "P012": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.30, 17.00, 11.00, 10.50],
    },
    "P013": {
        "start": [10.91, 12.03, 6.41, 5.03],
        "end": [20.00, 19.80, 11.00, 10.00],
    },
    "P014": {
        "start": [16.00, 12.00, 6.00, 17.00],
        "end": [20.20, 19.00, 12.00, 24.00],
    },
    "P015": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.20, 17.50, 11.00, 11.00],
    },
    "P016": {
        "start": [12.00, 15.00, 6.00, 6.00],
        "end": [17.06, 22.00, 11.00, 12.00],
    },
}


# =====================================================
# SAMPLE ENTROPY
# =====================================================

def sample_entropy(
    signal,
    m=EMBEDDING_DIMENSION,
    r_ratio=TOLERANCE_RATIO,
    min_length=MIN_SIGNAL_LENGTH,
):
    """
    Calculate sample entropy using Chebyshev distance.

    The same eligible starting positions are used for the
    length-m and length-(m+1) template comparisons.

    Parameters
    ----------
    signal : array-like
        One-dimensional signal.

    m : int
        Embedding dimension.

    r_ratio : float
        Tolerance expressed as a proportion of the signal
        standard deviation.

    min_length : int
        Minimum acceptable number of finite observations.

    Returns
    -------
    float
        Sample entropy value.

        Larger values indicate greater irregularity or
        lower predictability.

        Returns np.nan when the calculation cannot be
        completed.
    """

    signal = np.asarray(
        signal,
        dtype=float,
    )

    # Remove nonfinite values
    signal = signal[
        np.isfinite(signal)
    ]

    n = len(signal)

    if n < min_length:
        return np.nan

    if m < 1:
        raise ValueError(
            "Embedding dimension must be at least 1."
        )

    if r_ratio <= 0:
        raise ValueError(
            "Tolerance ratio must be greater than 0."
        )

    signal_sd = np.std(
        signal,
        ddof=1,
    )

    if (
        not np.isfinite(signal_sd)
        or signal_sd <= 0
    ):
        return np.nan

    tolerance = (
        r_ratio * signal_sd
    )

    # Use the same number of starting positions for
    # m-length and m+1-length templates.
    #
    # Starting indices:
    # 0 through n - m - 1
    #
    # Number of templates:
    # n - m
    number_of_templates = n - m

    if number_of_templates < 2:
        return np.nan

    templates_m = np.array(
        [
            signal[i:i + m]
            for i in range(
                number_of_templates
            )
        ],
        dtype=float,
    )

    templates_m1 = np.array(
        [
            signal[i:i + m + 1]
            for i in range(
                number_of_templates
            )
        ],
        dtype=float,
    )

    matches_m = 0
    matches_m1 = 0

    # Compare only unique template pairs:
    # j > i.
    #
    # This excludes self-matches and avoids counting
    # every pair twice.
    for i in range(
        number_of_templates - 1
    ):

        comparison_m = (
            templates_m[i + 1:]
            - templates_m[i]
        )

        distances_m = np.max(
            np.abs(comparison_m),
            axis=1,
        )

        comparison_m1 = (
            templates_m1[i + 1:]
            - templates_m1[i]
        )

        distances_m1 = np.max(
            np.abs(comparison_m1),
            axis=1,
        )

        matches_m += int(
            np.sum(
                distances_m <= tolerance
            )
        )

        matches_m1 += int(
            np.sum(
                distances_m1 <= tolerance
            )
        )

    if matches_m == 0:
        return np.nan

    if matches_m1 == 0:
        return np.nan

    sampen = -np.log(
        matches_m1 / matches_m
    )

    return float(sampen)


# =====================================================
# REPORTING FUNCTIONS
# =====================================================

def format_p(p):
    """
    Format p-values for manuscript reporting.
    """

    if pd.isna(p):
        return "NA"

    if p < 0.001:
        return "< .001"

    return (
        f"= {p:.3f}"
        .replace(
            "0.",
            ".",
        )
    )


def mean_sd_ci(values):
    """
    Return mean ± SD and the 95% confidence interval
    of the condition mean.
    """

    values = pd.Series(
        values,
        dtype=float,
    ).dropna()

    n = len(values)

    if n < 2:
        return "NA"

    mean_value = values.mean()
    sd_value = values.std(
        ddof=1
    )

    standard_error = (
        sd_value / np.sqrt(n)
    )

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_low = (
        mean_value
        - t_critical * standard_error
    )

    ci_high = (
        mean_value
        + t_critical * standard_error
    )

    return (
        f"{mean_value:.3f} ± "
        f"{sd_value:.3f}\n"
        f"[{ci_low:.3f}, "
        f"{ci_high:.3f}]"
    )


def paired_difference_ci(
    differences,
):
    """
    Return the mean paired difference and its
    95% confidence interval.

    Difference direction:
    HIGH minus LOW.
    """

    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    n = len(differences)

    if n < 2:
        return (
            np.nan,
            np.nan,
            np.nan,
        )

    mean_difference = (
        differences.mean()
    )

    sd_difference = (
        differences.std(
            ddof=1
        )
    )

    standard_error = (
        sd_difference
        / np.sqrt(n)
    )

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_low = (
        mean_difference
        - t_critical * standard_error
    )

    ci_high = (
        mean_difference
        + t_critical * standard_error
    )

    return (
        mean_difference,
        ci_low,
        ci_high,
    )


def cohen_dz(
    differences,
):
    """
    Cohen's dz for paired HIGH-minus-LOW
    differences.
    """

    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    if len(differences) < 2:
        return np.nan

    sd_difference = (
        differences.std(
            ddof=1
        )
    )

    if sd_difference == 0:
        return np.nan

    return (
        differences.mean()
        / sd_difference
    )


def matched_rank_biserial(
    differences,
):
    """
    Matched-pairs rank-biserial correlation.

    Positive values indicate larger values
    under HIGH.
    """

    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    # Wilcoxon excludes zero differences
    differences = differences[
        differences != 0
    ]

    if differences.empty:
        return np.nan

    ranks = stats.rankdata(
        np.abs(
            differences
        )
    )

    positive_rank_sum = ranks[
        differences.to_numpy() > 0
    ].sum()

    negative_rank_sum = ranks[
        differences.to_numpy() < 0
    ].sum()

    total_rank_sum = (
        positive_rank_sum
        + negative_rank_sum
    )

    if total_rank_sum == 0:
        return np.nan

    return (
        positive_rank_sum
        - negative_rank_sum
    ) / total_rank_sum


# =====================================================
# CALCULATE TRIAL-LEVEL SAMPLE ENTROPY
# =====================================================

trial_rows = []
missing_files = []
processing_errors = []

required_columns = [
    "CoM pos x",
    "CoM pos y",
    "CoM pos z",
    "CoM vel x",
    "CoM vel y",
    "CoM vel z",
    "CoM acc x",
    "CoM acc y",
    "CoM acc z",
]


for participant in participants:

    for condition, trials in (
        conditions.items()
    ):

        for trial in trials:

            file_name = (
                f"{participant}-"
                f"{trial:03d}.xlsx"
            )

            if not os.path.exists(
                file_name
            ):
                missing_files.append(
                    file_name
                )
                continue

            try:
                trial_data = (
                    pd.read_excel(
                        file_name,
                        sheet_name=(
                            "Center of Mass"
                        ),
                    )
                )

                trial_data.columns = (
                    trial_data.columns
                    .astype(str)
                    .str.strip()
                )

                missing_columns = [
                    column
                    for column
                    in required_columns
                    if column
                    not in trial_data.columns
                ]

                if missing_columns:
                    processing_errors.append(
                        {
                            "File": file_name,
                            "Error": (
                                "Missing columns: "
                                + ", ".join(
                                    missing_columns
                                )
                            ),
                        }
                    )
                    continue

                timing_index = (
                    trial - 3
                )

                if participant in cut_times:

                    start_time_seconds = (
                        cut_times[
                            participant
                        ][
                            timing_index
                        ]
                    )

                    start_frame = int(
                        round(
                            start_time_seconds
                            * fs
                        )
                    )

                    end_frame = len(
                        trial_data
                    )

                    end_time_seconds = (
                        end_frame / fs
                    )

                else:

                    start_time_seconds = (
                        reaction_times[
                            participant
                        ]["start"][
                            timing_index
                        ]
                    )

                    end_time_seconds = (
                        reaction_times[
                            participant
                        ]["end"][
                            timing_index
                        ]
                    )

                    start_frame = int(
                        round(
                            start_time_seconds
                            * fs
                        )
                    )

                    end_frame = int(
                        round(
                            end_time_seconds
                            * fs
                        )
                    )

                start_frame = max(
                    0,
                    start_frame,
                )

                end_frame = min(
                    len(trial_data),
                    end_frame,
                )

                if (
                    end_frame
                    <= start_frame
                ):
                    processing_errors.append(
                        {
                            "File": file_name,
                            "Error": (
                                "Invalid segment: "
                                f"start={start_frame}, "
                                f"end={end_frame}"
                            ),
                        }
                    )
                    continue

                segment = (
                    trial_data.iloc[
                        start_frame:
                        end_frame
                    ]
                    .copy()
                )

                segment_samples = len(
                    segment
                )

                segment_duration = (
                    segment_samples
                    / fs
                )

                position_magnitude = (
                    np.sqrt(
                        segment[
                            "CoM pos x"
                        ] ** 2
                        + segment[
                            "CoM pos y"
                        ] ** 2
                        + segment[
                            "CoM pos z"
                        ] ** 2
                    )
                    .to_numpy(
                        dtype=float
                    )
                )

                velocity_magnitude = (
                    np.sqrt(
                        segment[
                            "CoM vel x"
                        ] ** 2
                        + segment[
                            "CoM vel y"
                        ] ** 2
                        + segment[
                            "CoM vel z"
                        ] ** 2
                    )
                    .to_numpy(
                        dtype=float
                    )
                )

                acceleration_magnitude = (
                    np.sqrt(
                        segment[
                            "CoM acc x"
                        ] ** 2
                        + segment[
                            "CoM acc y"
                        ] ** 2
                        + segment[
                            "CoM acc z"
                        ] ** 2
                    )
                    .to_numpy(
                        dtype=float
                    )
                )

                position_valid_samples = int(
                    np.sum(
                        np.isfinite(
                            position_magnitude
                        )
                    )
                )

                velocity_valid_samples = int(
                    np.sum(
                        np.isfinite(
                            velocity_magnitude
                        )
                    )
                )

                acceleration_valid_samples = int(
                    np.sum(
                        np.isfinite(
                            acceleration_magnitude
                        )
                    )
                )

                position_sd = np.nanstd(
                    position_magnitude,
                    ddof=1,
                )

                velocity_sd = np.nanstd(
                    velocity_magnitude,
                    ddof=1,
                )

                acceleration_sd = np.nanstd(
                    acceleration_magnitude,
                    ddof=1,
                )

                trial_rows.append(
                    {
                        "Participant":
                            participant,
                        "Condition":
                            condition,
                        "Trial":
                            trial,
                        "File":
                            file_name,
                        "Sampling Frequency Hz":
                            fs,
                        "Start Time s":
                            start_time_seconds,
                        "End Time s":
                            end_time_seconds,
                        "Start Frame":
                            start_frame,
                        "End Frame":
                            end_frame,
                        "Segment Samples":
                            segment_samples,
                        "Segment Duration s":
                            segment_duration,
                        "Embedding Dimension":
                            EMBEDDING_DIMENSION,
                        "Tolerance Ratio":
                            TOLERANCE_RATIO,
                        "Minimum Signal Length":
                            MIN_SIGNAL_LENGTH,
                        "Position Valid Samples":
                            position_valid_samples,
                        "Velocity Valid Samples":
                            velocity_valid_samples,
                        "Acceleration Valid Samples":
                            acceleration_valid_samples,
                        "Position SD":
                            position_sd,
                        "Velocity SD":
                            velocity_sd,
                        "Acceleration SD":
                            acceleration_sd,
                        "Position Tolerance":
                            (
                                TOLERANCE_RATIO
                                * position_sd
                            ),
                        "Velocity Tolerance":
                            (
                                TOLERANCE_RATIO
                                * velocity_sd
                            ),
                        "Acceleration Tolerance":
                            (
                                TOLERANCE_RATIO
                                * acceleration_sd
                            ),
                        "SampEn_Position":
                            sample_entropy(
                                position_magnitude
                            ),
                        "SampEn_Velocity":
                            sample_entropy(
                                velocity_magnitude
                            ),
                        "SampEn_Acceleration":
                            sample_entropy(
                                acceleration_magnitude
                            ),
                    }
                )

            except Exception as error:

                processing_errors.append(
                    {
                        "File": file_name,
                        "Error": str(error),
                    }
                )


trial_level = pd.DataFrame(
    trial_rows
)

if trial_level.empty:
    raise ValueError(
        "No sample entropy values were "
        "calculated. Check file paths, "
        "file names, worksheets, and "
        "required columns."
    )


sample_long = trial_level.melt(
    id_vars=["Participant", "Condition", "Trial"],
    value_vars=["SampEn_Position", "SampEn_Velocity", "SampEn_Acceleration"],
    var_name="Metric",
    value_name="Value",
)
sample_long["Metric"] = sample_long["Metric"].map({
    "SampEn_Position": "Position",
    "SampEn_Velocity": "Velocity",
    "SampEn_Acceleration": "Acceleration",
})
metric_order = ["Position", "Velocity", "Acceleration"]
sample_long["Metric"] = pd.Categorical(
    sample_long["Metric"], categories=metric_order, ordered=True
)
sample_long = sample_long.sort_values(
    ["Metric", "Participant", "Condition", "Trial"]
).reset_index(drop=True)

run_repeated_family(
    sample_long,
    "Metric", "Value", "Participant", "Condition",
    ["LOW", "HIGH"],
    "Table_Sample_Entropy.xlsx",
    expected_repetitions=2,
    decimals=3,
)



Saved: Statistical_Output\Table_Sample_Entropy.xlsx


,Metric,Participants,LOW,HIGH,Primary Test,Significance
0,Position,16,"0.012 ± 0.008\n[0.008, 0.017]","0.024 ± 0.018\n[0.014, 0.033]",Wilcoxon signed-rank,W = 9.000; p = .001; q = .002; r_rb = 0.868
1,Velocity,16,"0.040 ± 0.026\n[0.027, 0.054]","0.084 ± 0.059\n[0.052, 0.115]",Paired-samples t-test,t(15) = 4.044; p = .001; q = .002; dz = 1.011
2,Acceleration,16,"0.284 ± 0.141\n[0.209, 0.359]","0.443 ± 0.220\n[0.326, 0.561]",Paired-samples t-test,t(15) = 2.967; p = .010; q = .010; dz = 0.742


(         Metric  Participants                            LOW  \
 0      Position            16  0.012 ± 0.008\n[0.008, 0.017]   
 1      Velocity            16  0.040 ± 0.026\n[0.027, 0.054]   
 2  Acceleration            16  0.284 ± 0.141\n[0.209, 0.359]   
 
                             HIGH           Primary Test  \
 0  0.024 ± 0.018\n[0.014, 0.033]   Wilcoxon signed-rank   
 1  0.084 ± 0.059\n[0.052, 0.115]  Paired-samples t-test   
 2  0.443 ± 0.220\n[0.326, 0.561]  Paired-samples t-test   
 
                                     Significance  
 0    W = 9.000; p = .001; q = .002; r_rb = 0.868  
 1  t(15) = 4.044; p = .001; q = .002; dz = 1.011  
 2  t(15) = 2.967; p = .010; q = .010; dz = 0.742  ,
          Metric  Participants                            LOW  \
 0      Position            16  0.012 ± 0.008\n[0.008, 0.017]   
 1      Velocity            16  0.040 ± 0.026\n[0.027, 0.054]   
 2  Acceleration            16  0.284 ± 0.141\n[0.209, 0.359]   
 
                         

## Table: Lyapunov Exponent

In [6]:
# =====================================================
# PARAMETERS
# =====================================================

fs = 60  # Sampling frequency in Hz

# Rosenstein largest Lyapunov exponent parameters
embedding_dimension = 5
time_delay_samples = 10          # delay-embedding lag
theiler_window_samples = 60      # exclude neighbors within ±1.0 s
trajectory_length_samples = 20   # divergence trajectory length
fit_offset_samples = 0           # first divergence point included in fit
fit_method = "poly"              # ordinary least-squares line fit

participants = [f"P{i:03d}" for i in range(1, 17)]

conditions = {
    "LOW": [3, 4],
    "HIGH": [5, 6],
}


# =====================================================
# TIMING DICTIONARIES, SECONDS
# =====================================================

cut_times = {
    "P001": [12.55, 15.17, 6.90, 7.56],
    "P002": [15.81, 15.86, 7.40, 6.00],
}

reaction_times = {
    "P003": {
        "start": [10.28, 13.53, 7.50, 8.15],
        "end": [21.50, 23.80, 13.50, 12.50],
    },
    "P004": {
        "start": [8.31, 12.50, 5.88, 5.18],
        "end": [18.00, 22.00, 11.40, 9.80],
    },
    "P005": {
        "start": [12.16, 11.78, 5.58, 6.03],
        "end": [20.60, 20.80, 10.00, 10.50],
    },
    "P006": {
        "start": [10.66, 12.90, 5.81, 6.25],
        "end": [19.70, 20.50, 7.20, 9.80],
    },
    "P007": {
        "start": [10.00, 12.00, 6.10, 6.00],
        "end": [23.70, 15.60, 8.00, 9.60],
    },
    "P008": {
        "start": [11.00, 12.00, 6.00, 7.00],
        "end": [19.60, 24.00, 11.80, 10.20],
    },
    "P009": {
        "start": [10.29, 10.56, 5.86, 6.02],
        "end": [20.50, 23.00, 9.00, 12.00],
    },
    "P010": {
        "start": [12.00, 12.00, 6.00, 16.50],
        "end": [15.50, 16.20, 11.00, 20.00],
    },
    "P011": {
        "start": [10.94, 12.76, 6.10, 7.91],
        "end": [20.80, 24.00, 9.30, 14.00],
    },
    "P012": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.30, 17.00, 11.00, 10.50],
    },
    "P013": {
        "start": [10.91, 12.03, 6.41, 5.03],
        "end": [20.00, 19.80, 11.00, 10.00],
    },
    "P014": {
        "start": [16.00, 12.00, 6.00, 17.00],
        "end": [20.20, 19.00, 12.00, 24.00],
    },
    "P015": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.20, 17.50, 11.00, 11.00],
    },
    "P016": {
        "start": [12.00, 15.00, 6.00, 6.00],
        "end": [17.06, 22.00, 11.00, 12.00],
    },
}


# =====================================================
# LARGEST LYAPUNOV EXPONENT: ROSENSTEIN METHOD
# =====================================================

def largest_lyapunov(
    signal,
    m=embedding_dimension,
    lag=time_delay_samples,
    fs=fs,
    min_tsep=theiler_window_samples,
    trajectory_len=trajectory_length_samples,
    fit_offset=fit_offset_samples,
    fit=fit_method,
):
    """
    Estimate the largest Lyapunov exponent with the Rosenstein method
    using the validated nolds.lyap_r implementation.

    Parameters
    ----------
    signal : array-like
        One-dimensional time series.
    m : int
        Embedding dimension.
    lag : int
        Delay between coordinates of each embedded vector, in samples.
    fs : float
        Sampling frequency in Hz.
    min_tsep : int
        Theiler window/minimum temporal separation between neighboring
        state-space vectors, in samples.
    trajectory_len : int
        Number of forward samples used to construct the mean logarithmic
        divergence trajectory.
    fit_offset : int
        Number of initial divergence samples excluded from the linear fit.
    fit : {"poly", "RANSAC"}
        Line-fitting method. "poly" gives an ordinary least-squares fit.

    Returns
    -------
    float
        Largest Lyapunov exponent in s^-1. Returns np.nan when the signal
        is too short or the estimate cannot be computed.
    """
    signal = np.asarray(signal, dtype=float)
    signal = signal[np.isfinite(signal)]

    if len(signal) == 0 or fs <= 0:
        return np.nan

    if np.nanstd(signal, ddof=1) == 0:
        return np.nan

    # nolds reports the minimum length implied by the selected embedding,
    # Theiler window, and divergence-trajectory settings.
    minimum_required = nolds.lyap_r_len(
        emb_dim=m,
        lag=lag,
        min_tsep=min_tsep,
        trajectory_len=trajectory_len,
    )

    if len(signal) < minimum_required:
        return np.nan

    try:
        exponent = nolds.lyap_r(
            signal,
            emb_dim=m,
            lag=lag,
            min_tsep=min_tsep,
            tau=1.0 / fs,          # converts slope from per-sample to s^-1
            trajectory_len=trajectory_len,
            fit=fit,
            fit_offset=fit_offset,
            debug_plot=False,
            debug_data=False,
        )
    except (ValueError, RuntimeError, FloatingPointError):
        return np.nan

    return float(exponent) if np.isfinite(exponent) else np.nan


def lyapunov_debug_data(
    signal,
    m=embedding_dimension,
    lag=time_delay_samples,
    fs=fs,
    min_tsep=theiler_window_samples,
    trajectory_len=trajectory_length_samples,
    fit_offset=fit_offset_samples,
    fit=fit_method,
):
    """
    Return the LLE and divergence-curve data for diagnostic plotting.

    Returns
    -------
    exponent : float
    time_seconds : ndarray
    mean_log_divergence : ndarray
    fitted_line : ndarray
    """
    signal = np.asarray(signal, dtype=float)
    signal = signal[np.isfinite(signal)]

    minimum_required = nolds.lyap_r_len(
        emb_dim=m,
        lag=lag,
        min_tsep=min_tsep,
        trajectory_len=trajectory_len,
    )

    if len(signal) < minimum_required:
        return np.nan, np.array([]), np.array([]), np.array([])

    try:
        exponent, debug = nolds.lyap_r(
            signal,
            emb_dim=m,
            lag=lag,
            min_tsep=min_tsep,
            tau=1.0 / fs,
            trajectory_len=trajectory_len,
            fit=fit,
            fit_offset=fit_offset,
            debug_plot=False,
            debug_data=True,
        )
    except (ValueError, RuntimeError, FloatingPointError):
        return np.nan, np.array([]), np.array([]), np.array([])

    ks, divergence, coefficients = debug
    time_seconds = np.asarray(ks, dtype=float) / fs
    divergence = np.asarray(divergence, dtype=float)
    fitted_line = np.polyval(coefficients, np.asarray(ks, dtype=float))

    return float(exponent), time_seconds, divergence, fitted_line


# =====================================================
# REPORTING FUNCTIONS
# =====================================================

def format_p(p):
    if pd.isna(p):
        return "NA"

    if p < 0.001:
        return "< .001"

    return f"= {p:.3f}".replace("0.", ".")


def mean_sd_ci(values):
    """
    Return mean ± SD and 95% confidence interval.
    """

    values = pd.Series(
        values,
        dtype=float,
    ).dropna()

    n = len(values)

    if n < 2:
        return "NA"

    mean = values.mean()
    sd = values.std(ddof=1)
    se = sd / np.sqrt(n)

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_low = mean - t_critical * se
    ci_high = mean + t_critical * se

    return (
        f"{mean:.3f} ± {sd:.3f}\n"
        f"[{ci_low:.3f}, {ci_high:.3f}]"
    )


def cohen_dz(differences):
    """
    Cohen's dz for paired HIGH-minus-LOW differences.
    """

    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    if len(differences) < 2:
        return np.nan

    sd_difference = differences.std(ddof=1)

    if sd_difference == 0:
        return np.nan

    return (
        differences.mean()
        / sd_difference
    )


def matched_rank_biserial(differences):
    """
    Matched-pairs rank-biserial correlation.

    Positive values indicate larger values under HIGH.
    """

    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    differences = differences[
        differences != 0
    ]

    if differences.empty:
        return np.nan

    ranks = stats.rankdata(
        np.abs(differences)
    )

    positive_rank_sum = ranks[
        differences.to_numpy() > 0
    ].sum()

    negative_rank_sum = ranks[
        differences.to_numpy() < 0
    ].sum()

    total_rank_sum = (
        positive_rank_sum
        + negative_rank_sum
    )

    if total_rank_sum == 0:
        return np.nan

    return (
        positive_rank_sum
        - negative_rank_sum
    ) / total_rank_sum


# =====================================================
# COMPUTE TRIAL-LEVEL LLE
# =====================================================

trial_rows = []

required_columns = [
    "CoM pos x",
    "CoM pos y",
    "CoM pos z",
    "CoM vel x",
    "CoM vel y",
    "CoM vel z",
    "CoM acc x",
    "CoM acc y",
    "CoM acc z",
]

for participant in participants:

    for condition, trials in conditions.items():

        for trial in trials:

            file_name = (
                f"{participant}-{trial:03d}.xlsx"
            )

            if not os.path.exists(file_name):
                continue

            try:
                trial_data = pd.read_excel(
                    file_name,
                    sheet_name="Center of Mass",
                )

                trial_data.columns = (
                    trial_data.columns
                    .astype(str)
                    .str.strip()
                )

                missing_columns = [
                    column
                    for column in required_columns
                    if column not in trial_data.columns
                ]

                if missing_columns:
                    continue

                timing_index = trial - 3

                if participant in cut_times:
                    start_time_seconds = (
                        cut_times[participant][
                            timing_index
                        ]
                    )

                    start_frame = int(
                        round(
                            start_time_seconds * fs
                        )
                    )

                    end_frame = len(trial_data)
                    end_time_seconds = end_frame / fs

                else:
                    start_time_seconds = (
                        reaction_times[participant][
                            "start"
                        ][timing_index]
                    )

                    end_time_seconds = (
                        reaction_times[participant][
                            "end"
                        ][timing_index]
                    )

                    start_frame = int(
                        round(
                            start_time_seconds * fs
                        )
                    )

                    end_frame = int(
                        round(
                            end_time_seconds * fs
                        )
                    )

                start_frame = max(
                    0,
                    start_frame,
                )

                end_frame = min(
                    len(trial_data),
                    end_frame,
                )

                if end_frame <= start_frame:
                    continue

                segment = trial_data.iloc[
                    start_frame:end_frame
                ].copy()

                # -------------------------------------
                # DATA-LENGTH INFORMATION
                # -------------------------------------

                segment_length_samples = len(segment)

                segment_duration_seconds = (
                    segment_length_samples / fs
                )

                embedded_length = (
                    segment_length_samples
                    - (embedding_dimension - 1) * time_delay_samples
                )

                minimum_required_samples = nolds.lyap_r_len(
                    emb_dim=embedding_dimension,
                    lag=time_delay_samples,
                    min_tsep=theiler_window_samples,
                    trajectory_len=trajectory_length_samples,
                )

                if embedded_length <= 0:
                    continue

                position_magnitude = np.sqrt(
                    segment["CoM pos x"] ** 2
                    + segment["CoM pos y"] ** 2
                    + segment["CoM pos z"] ** 2
                )

                velocity_magnitude = np.sqrt(
                    segment["CoM vel x"] ** 2
                    + segment["CoM vel y"] ** 2
                    + segment["CoM vel z"] ** 2
                )

                acceleration_magnitude = np.sqrt(
                    segment["CoM acc x"] ** 2
                    + segment["CoM acc y"] ** 2
                    + segment["CoM acc z"] ** 2
                )

                position_valid_samples = (
                    np.isfinite(
                        position_magnitude
                    ).sum()
                )

                velocity_valid_samples = (
                    np.isfinite(
                        velocity_magnitude
                    ).sum()
                )

                acceleration_valid_samples = (
                    np.isfinite(
                        acceleration_magnitude
                    ).sum()
                )

                trial_rows.append(
                    {
                        "Participant": participant,
                        "Condition": condition,
                        "Trial": trial,
                        "File": file_name,
                        "Sampling_Frequency_Hz": fs,
                        "Start_Time_Seconds":
                            start_time_seconds,
                        "End_Time_Seconds":
                            end_time_seconds,
                        "Start_Frame": start_frame,
                        "End_Frame": end_frame,
                        "Segment_Length_Samples":
                            segment_length_samples,
                        "Segment_Duration_Seconds":
                            segment_duration_seconds,
                        "Embedding_Dimension":
                            embedding_dimension,
                        "Time_Delay_Samples":
                            time_delay_samples,
                        "Time_Delay_Seconds":
                            time_delay_samples / fs,
                        "Embedded_Length":
                            embedded_length,
                        "Theiler_Window_Samples":
                            theiler_window_samples,
                        "Theiler_Window_Seconds":
                            theiler_window_samples / fs,
                        "Trajectory_Length_Samples":
                            trajectory_length_samples,
                        "Trajectory_Length_Seconds":
                            trajectory_length_samples / fs,
                        "Fit_Offset_Samples":
                            fit_offset_samples,
                        "Fit_Method":
                            fit_method,
                        "Minimum_Required_Samples":
                            minimum_required_samples,
                        "Position_Valid_Samples":
                            position_valid_samples,
                        "Velocity_Valid_Samples":
                            velocity_valid_samples,
                        "Acceleration_Valid_Samples":
                            acceleration_valid_samples,
                        "Lyap_CoM_Pos":
                            largest_lyapunov(
                                position_magnitude,
                                m=embedding_dimension,
                                lag=time_delay_samples,
                                fs=fs,
                                min_tsep=theiler_window_samples,
                                trajectory_len=trajectory_length_samples,
                                fit_offset=fit_offset_samples,
                                fit=fit_method,
                            ),
                        "Lyap_CoM_Vel":
                            largest_lyapunov(
                                velocity_magnitude,
                                m=embedding_dimension,
                                lag=time_delay_samples,
                                fs=fs,
                                min_tsep=theiler_window_samples,
                                trajectory_len=trajectory_length_samples,
                                fit_offset=fit_offset_samples,
                                fit=fit_method,
                            ),
                        "Lyap_CoM_Acc":
                            largest_lyapunov(
                                acceleration_magnitude,
                                m=embedding_dimension,
                                lag=time_delay_samples,
                                fs=fs,
                                min_tsep=theiler_window_samples,
                                trajectory_len=trajectory_length_samples,
                                fit_offset=fit_offset_samples,
                                fit=fit_method,
                            ),
                    }
                )

            except Exception as error:
                continue


trial_level = pd.DataFrame(trial_rows)

if trial_level.empty:
    raise ValueError(
        "No LLE data were computed. "
        "Check the working directory and "
        "input workbook structure."
    )


lle_long = trial_level.melt(
    id_vars=["Participant", "Condition", "Trial"],
    value_vars=["Lyap_CoM_Pos", "Lyap_CoM_Vel", "Lyap_CoM_Acc"],
    var_name="Metric",
    value_name="Value",
)
lle_long["Metric"] = lle_long["Metric"].map({
    "Lyap_CoM_Pos": "Position",
    "Lyap_CoM_Vel": "Velocity",
    "Lyap_CoM_Acc": "Acceleration",
})
metric_order = ["Position", "Velocity", "Acceleration"]
lle_long["Metric"] = pd.Categorical(
    lle_long["Metric"], categories=metric_order, ordered=True
)
lle_long = lle_long.sort_values(
    ["Metric", "Participant", "Condition", "Trial"]
).reset_index(drop=True)

run_repeated_family(
    lle_long,
    "Metric", "Value", "Participant", "Condition",
    ["LOW", "HIGH"],
    "Table_Lyapunov_Exponent.xlsx",
    expected_repetitions=2,
    decimals=3,
)



Saved: Statistical_Output\Table_Lyapunov_Exponent.xlsx


,Metric,Participants,LOW,HIGH,Primary Test,Significance
0,Position,16,"1.927 ± 0.732\n[1.537, 2.317]","1.502 ± 0.912\n[1.016, 1.989]",Paired-samples t-test,t(15) = -1.948; p = .070; q = .102; dz = -0.487
1,Velocity,16,"2.183 ± 0.552\n[1.889, 2.477]","1.827 ± 0.765\n[1.419, 2.235]",Paired-samples t-test,t(15) = -1.741; p = .102; q = .102; dz = -0.435
2,Acceleration,16,"1.584 ± 0.330\n[1.408, 1.760]","1.193 ± 0.636\n[0.854, 1.532]",Paired-samples t-test,t(15) = -2.263; p = .039; q = .102; dz = -0.566


(         Metric  Participants                            LOW  \
 0      Position            16  1.927 ± 0.732\n[1.537, 2.317]   
 1      Velocity            16  2.183 ± 0.552\n[1.889, 2.477]   
 2  Acceleration            16  1.584 ± 0.330\n[1.408, 1.760]   
 
                             HIGH           Primary Test  \
 0  1.502 ± 0.912\n[1.016, 1.989]  Paired-samples t-test   
 1  1.827 ± 0.765\n[1.419, 2.235]  Paired-samples t-test   
 2  1.193 ± 0.636\n[0.854, 1.532]  Paired-samples t-test   
 
                                       Significance  
 0  t(15) = -1.948; p = .070; q = .102; dz = -0.487  
 1  t(15) = -1.741; p = .102; q = .102; dz = -0.435  
 2  t(15) = -2.263; p = .039; q = .102; dz = -0.566  ,
          Metric  Participants                            LOW  \
 0      Position            16  1.927 ± 0.732\n[1.537, 2.317]   
 1      Velocity            16  2.183 ± 0.552\n[1.889, 2.477]   
 2  Acceleration            16  1.584 ± 0.330\n[1.408, 1.760]   
 
                 

## Table: Smoothness (SPARC)

In [8]:
# =====================================================
# PARAMETERS
# =====================================================

fs = 60  # Sampling frequency, Hz

participants = [
    f"P{i:03d}"
    for i in range(1, 17)
]

conditions = {
    "LOW": [3, 4],
    "HIGH": [5, 6],
}

# Published SPARC reference settings
SPARC_PADLEVEL = 4
SPARC_MAX_CUTOFF_HZ = 10.0
SPARC_AMPLITUDE_THRESHOLD = 0.05
MIN_SIGNAL_LENGTH = 60


# =====================================================
# TIMING DICTIONARIES, SECONDS
# =====================================================

cut_times = {
    "P001": [12.55, 15.17, 6.90, 7.56],
    "P002": [15.81, 15.86, 7.40, 6.00],
}

reaction_times = {
    "P003": {
        "start": [10.28, 13.53, 7.50, 8.15],
        "end": [21.50, 23.80, 13.50, 12.50],
    },
    "P004": {
        "start": [8.31, 12.50, 5.88, 5.18],
        "end": [18.00, 22.00, 11.40, 9.80],
    },
    "P005": {
        "start": [12.16, 11.78, 5.58, 6.03],
        "end": [20.60, 20.80, 10.00, 10.50],
    },
    "P006": {
        "start": [10.66, 12.90, 5.81, 6.25],
        "end": [19.70, 20.50, 7.20, 9.80],
    },
    "P007": {
        "start": [10.00, 12.00, 6.10, 6.00],
        "end": [23.70, 15.60, 8.00, 9.60],
    },
    "P008": {
        "start": [11.00, 12.00, 6.00, 7.00],
        "end": [19.60, 24.00, 11.80, 10.20],
    },
    "P009": {
        "start": [10.29, 10.56, 5.86, 6.02],
        "end": [20.50, 23.00, 9.00, 12.00],
    },
    "P010": {
        "start": [12.00, 12.00, 6.00, 16.50],
        "end": [15.50, 16.20, 11.00, 20.00],
    },
    "P011": {
        "start": [10.94, 12.76, 6.10, 7.91],
        "end": [20.80, 24.00, 9.30, 14.00],
    },
    "P012": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.30, 17.00, 11.00, 10.50],
    },
    "P013": {
        "start": [10.91, 12.03, 6.41, 5.03],
        "end": [20.00, 19.80, 11.00, 10.00],
    },
    "P014": {
        "start": [16.00, 12.00, 6.00, 17.00],
        "end": [20.20, 19.00, 12.00, 24.00],
    },
    "P015": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.20, 17.50, 11.00, 11.00],
    },
    "P016": {
        "start": [12.00, 15.00, 6.00, 6.00],
        "end": [17.06, 22.00, 11.00, 12.00],
    },
}


# =====================================================
# SPARC SMOOTHNESS
# =====================================================

def sparc(
    movement,
    fs,
    padlevel=SPARC_PADLEVEL,
    max_cutoff_hz=SPARC_MAX_CUTOFF_HZ,
    amplitude_threshold=SPARC_AMPLITUDE_THRESHOLD,
    min_length=MIN_SIGNAL_LENGTH,
):
    """
    Calculate SPARC movement smoothness using the published
    reference implementation.

    Parameters
    ----------
    movement : array-like
        One-dimensional movement profile. A speed profile is
        most consistent with the original SPARC formulation.

    fs : float
        Sampling frequency in Hz.

    padlevel : int
        Zero-padding level. The FFT length equals the next
        power of two multiplied by 2**padlevel.

    max_cutoff_hz : float
        Maximum frequency considered in the calculation.

    amplitude_threshold : float
        Normalized magnitude-spectrum threshold used to select
        the adaptive cutoff frequency.

    min_length : int
        Minimum number of finite observations.

    Returns
    -------
    sparc_value : float
        SPARC smoothness value. Values closer to zero indicate
        smoother movement. More negative values indicate less
        smooth movement.

    cutoff_frequency : float
        Adaptive cutoff frequency used for the arc-length
        calculation.

    selected_bins : int
        Number of spectral bins used in the calculation.

    nfft : int
        FFT length after zero padding.
    """

    movement = np.asarray(
        movement,
        dtype=float,
    )

    movement = movement[
        np.isfinite(movement)
    ]

    if len(movement) < min_length:
        return np.nan, np.nan, 0, np.nan

    if fs <= 0:
        raise ValueError(
            "Sampling frequency must be greater than zero."
        )

    if padlevel < 0:
        raise ValueError(
            "SPARC pad level cannot be negative."
        )

    if (
        max_cutoff_hz <= 0
        or max_cutoff_hz > fs / 2
    ):
        raise ValueError(
            "The maximum cutoff frequency must be greater "
            "than zero and no greater than the Nyquist frequency."
        )

    if not 0 < amplitude_threshold < 1:
        raise ValueError(
            "The SPARC amplitude threshold must be between "
            "zero and one."
        )

    # Do not mean-center the signal. The published reference
    # implementation calculates SPARC directly from the
    # movement profile.

    # FFT length with zero padding
    nfft = int(
        2 ** (
            np.ceil(
                np.log2(
                    len(movement)
                )
            )
            + padlevel
        )
    )

    # Frequency vector used by the reference implementation
    frequencies = np.arange(
        0,
        fs,
        fs / nfft,
    )

    # Magnitude spectrum
    magnitude_spectrum = np.abs(
        np.fft.fft(
            movement,
            n=nfft,
        )
    )

    maximum_magnitude = np.max(
        magnitude_spectrum
    )

    if (
        not np.isfinite(maximum_magnitude)
        or maximum_magnitude <= 0
    ):
        return np.nan, np.nan, 0, nfft

    # Normalize magnitude spectrum
    normalized_spectrum = (
        magnitude_spectrum
        / maximum_magnitude
    )

    # First restrict the spectrum to the maximum cutoff
    maximum_cutoff_indices = np.where(
        frequencies <= max_cutoff_hz
    )[0]

    if len(maximum_cutoff_indices) < 2:
        return np.nan, np.nan, 0, nfft

    selected_frequencies = frequencies[
        maximum_cutoff_indices
    ]

    selected_spectrum = normalized_spectrum[
        maximum_cutoff_indices
    ]

    # Identify all bins at or above the amplitude threshold
    above_threshold_indices = np.where(
        selected_spectrum
        >= amplitude_threshold
    )[0]

    if len(above_threshold_indices) < 2:
        return np.nan, np.nan, 0, nfft

    # Published implementation retains the interval from the
    # first through the final above-threshold spectral bin.
    first_index = above_threshold_indices[0]
    last_index = above_threshold_indices[-1]

    selected_frequencies = selected_frequencies[
        first_index:last_index + 1
    ]

    selected_spectrum = selected_spectrum[
        first_index:last_index + 1
    ]

    if len(selected_frequencies) < 2:
        return np.nan, np.nan, 0, nfft

    frequency_range = (
        selected_frequencies[-1]
        - selected_frequencies[0]
    )

    if frequency_range <= 0:
        return np.nan, np.nan, 0, nfft

    # Normalize the frequency axis and calculate the negative
    # spectral arc length.
    normalized_frequency_difference = (
        np.diff(selected_frequencies)
        / frequency_range
    )

    spectrum_difference = np.diff(
        selected_spectrum
    )

    arc_length_elements = np.sqrt(
        normalized_frequency_difference ** 2
        + spectrum_difference ** 2
    )

    sparc_value = -np.sum(
        arc_length_elements
    )

    cutoff_frequency = (
        selected_frequencies[-1]
    )

    selected_bins = len(
        selected_frequencies
    )

    return (
        float(sparc_value),
        float(cutoff_frequency),
        int(selected_bins),
        int(nfft),
    )


# =====================================================
# REPORTING FUNCTIONS
# =====================================================

def format_p(p):
    if pd.isna(p):
        return "NA"

    if p < 0.001:
        return "< .001"

    return f"= {p:.3f}".replace(
        "0.",
        ".",
    )


def mean_sd_ci(values):
    """
    Return mean ± SD and 95% CI of the condition mean.
    """

    values = pd.Series(
        values,
        dtype=float,
    ).dropna()

    n = len(values)

    if n < 2:
        return "NA"

    mean_value = values.mean()
    sd_value = values.std(
        ddof=1
    )

    standard_error = (
        sd_value
        / np.sqrt(n)
    )

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_low = (
        mean_value
        - t_critical * standard_error
    )

    ci_high = (
        mean_value
        + t_critical * standard_error
    )

    return (
        f"{mean_value:.3f} ± "
        f"{sd_value:.3f}\n"
        f"[{ci_low:.3f}, "
        f"{ci_high:.3f}]"
    )


def paired_difference_ci(differences):
    """
    Return mean HIGH-minus-LOW difference and 95% CI.
    """

    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    n = len(differences)

    if n < 2:
        return np.nan, np.nan, np.nan

    mean_difference = differences.mean()

    sd_difference = differences.std(
        ddof=1
    )

    standard_error = (
        sd_difference
        / np.sqrt(n)
    )

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_low = (
        mean_difference
        - t_critical * standard_error
    )

    ci_high = (
        mean_difference
        + t_critical * standard_error
    )

    return (
        mean_difference,
        ci_low,
        ci_high,
    )


def cohen_dz(differences):
    """
    Cohen's dz for paired HIGH-minus-LOW differences.
    """

    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    if len(differences) < 2:
        return np.nan

    sd_difference = differences.std(
        ddof=1
    )

    if sd_difference == 0:
        return np.nan

    return (
        differences.mean()
        / sd_difference
    )


def matched_rank_biserial(differences):
    """
    Matched-pairs rank-biserial correlation.

    Positive values indicate larger values under HIGH.
    """

    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    differences = differences[
        differences != 0
    ]

    if differences.empty:
        return np.nan

    ranks = stats.rankdata(
        np.abs(
            differences
        )
    )

    positive_rank_sum = ranks[
        differences.to_numpy() > 0
    ].sum()

    negative_rank_sum = ranks[
        differences.to_numpy() < 0
    ].sum()

    total_rank_sum = (
        positive_rank_sum
        + negative_rank_sum
    )

    if total_rank_sum == 0:
        return np.nan

    return (
        positive_rank_sum
        - negative_rank_sum
    ) / total_rank_sum


# =====================================================
# COMPUTE TRIAL-LEVEL SPARC
# =====================================================

trial_rows = []
missing_files = []
processing_errors = []

required_columns = [
    "CoM pos x",
    "CoM pos y",
    "CoM pos z",
    "CoM vel x",
    "CoM vel y",
    "CoM vel z",
    "CoM acc x",
    "CoM acc y",
    "CoM acc z",
]


for participant in participants:

    for condition, trials in conditions.items():

        for trial in trials:

            file_name = (
                f"{participant}-"
                f"{trial:03d}.xlsx"
            )

            if not os.path.exists(file_name):
                missing_files.append(
                    file_name
                )
                continue

            try:
                trial_data = pd.read_excel(
                    file_name,
                    sheet_name="Center of Mass",
                )

                trial_data.columns = (
                    trial_data.columns
                    .astype(str)
                    .str.strip()
                )

                missing_columns = [
                    column
                    for column in required_columns
                    if column not in trial_data.columns
                ]

                if missing_columns:
                    processing_errors.append(
                        {
                            "File": file_name,
                            "Error": (
                                "Missing columns: "
                                + ", ".join(
                                    missing_columns
                                )
                            ),
                        }
                    )
                    continue

                timing_index = trial - 3

                if participant in cut_times:

                    start_time_seconds = (
                        cut_times[
                            participant
                        ][timing_index]
                    )

                    start_frame = int(
                        round(
                            start_time_seconds
                            * fs
                        )
                    )

                    end_frame = len(
                        trial_data
                    )

                    end_time_seconds = (
                        end_frame / fs
                    )

                else:

                    start_time_seconds = (
                        reaction_times[
                            participant
                        ]["start"][timing_index]
                    )

                    end_time_seconds = (
                        reaction_times[
                            participant
                        ]["end"][timing_index]
                    )

                    start_frame = int(
                        round(
                            start_time_seconds
                            * fs
                        )
                    )

                    end_frame = int(
                        round(
                            end_time_seconds
                            * fs
                        )
                    )

                start_frame = max(
                    0,
                    start_frame,
                )

                end_frame = min(
                    len(trial_data),
                    end_frame,
                )

                if end_frame <= start_frame:
                    processing_errors.append(
                        {
                            "File": file_name,
                            "Error": (
                                "Invalid segment: "
                                f"start={start_frame}, "
                                f"end={end_frame}"
                            ),
                        }
                    )
                    continue

                segment = trial_data.iloc[
                    start_frame:end_frame
                ].copy()

                segment_samples = len(
                    segment
                )

                segment_duration_seconds = (
                    segment_samples / fs
                )

                position_magnitude = np.sqrt(
                    segment["CoM pos x"] ** 2
                    + segment["CoM pos y"] ** 2
                    + segment["CoM pos z"] ** 2
                ).to_numpy(
                    dtype=float
                )

                velocity_magnitude = np.sqrt(
                    segment["CoM vel x"] ** 2
                    + segment["CoM vel y"] ** 2
                    + segment["CoM vel z"] ** 2
                ).to_numpy(
                    dtype=float
                )

                acceleration_magnitude = np.sqrt(
                    segment["CoM acc x"] ** 2
                    + segment["CoM acc y"] ** 2
                    + segment["CoM acc z"] ** 2
                ).to_numpy(
                    dtype=float
                )

                (
                    position_sparc,
                    position_cutoff,
                    position_bins,
                    position_nfft,
                ) = sparc(
                    position_magnitude,
                    fs=fs,
                )

                (
                    velocity_sparc,
                    velocity_cutoff,
                    velocity_bins,
                    velocity_nfft,
                ) = sparc(
                    velocity_magnitude,
                    fs=fs,
                )

                (
                    acceleration_sparc,
                    acceleration_cutoff,
                    acceleration_bins,
                    acceleration_nfft,
                ) = sparc(
                    acceleration_magnitude,
                    fs=fs,
                )

                trial_rows.append(
                    {
                        "Participant":
                            participant,
                        "Condition":
                            condition,
                        "Trial":
                            trial,
                        "File":
                            file_name,
                        "Sampling Frequency Hz":
                            fs,
                        "Start Time s":
                            start_time_seconds,
                        "End Time s":
                            end_time_seconds,
                        "Start Frame":
                            start_frame,
                        "End Frame":
                            end_frame,
                        "Segment Samples":
                            segment_samples,
                        "Segment Duration s":
                            segment_duration_seconds,
                        "SPARC Pad Level":
                            SPARC_PADLEVEL,
                        "Maximum Cutoff Hz":
                            SPARC_MAX_CUTOFF_HZ,
                        "Amplitude Threshold":
                            SPARC_AMPLITUDE_THRESHOLD,
                        "SPARC_Position":
                            position_sparc,
                        "Position Cutoff Hz":
                            position_cutoff,
                        "Position Selected Bins":
                            position_bins,
                        "Position NFFT":
                            position_nfft,
                        "SPARC_Velocity":
                            velocity_sparc,
                        "Velocity Cutoff Hz":
                            velocity_cutoff,
                        "Velocity Selected Bins":
                            velocity_bins,
                        "Velocity NFFT":
                            velocity_nfft,
                        "SPARC_Acceleration":
                            acceleration_sparc,
                        "Acceleration Cutoff Hz":
                            acceleration_cutoff,
                        "Acceleration Selected Bins":
                            acceleration_bins,
                        "Acceleration NFFT":
                            acceleration_nfft,
                    }
                )

            except Exception as error:
                processing_errors.append(
                    {
                        "File": file_name,
                        "Error": str(error),
                    }
                )


trial_level = pd.DataFrame(
    trial_rows
)

if trial_level.empty:
    raise ValueError(
        "No SPARC values were calculated. "
        "Check file paths, worksheets, and column names."
    )


sparc_long = trial_level.melt(
    id_vars=["Participant", "Condition", "Trial"],
    value_vars=["SPARC_Position", "SPARC_Velocity", "SPARC_Acceleration"],
    var_name="Metric",
    value_name="Value",
)
sparc_long["Metric"] = sparc_long["Metric"].map({
    "SPARC_Position": "Position",
    "SPARC_Velocity": "Velocity",
    "SPARC_Acceleration": "Acceleration",
})
metric_order = ["Position", "Velocity", "Acceleration"]
sparc_long["Metric"] = pd.Categorical(
    sparc_long["Metric"], categories=metric_order, ordered=True
)
sparc_long = sparc_long.sort_values(
    ["Metric", "Participant", "Condition", "Trial"]
).reset_index(drop=True)

run_repeated_family(
    sparc_long,
    "Metric", "Value", "Participant", "Condition",
    ["LOW", "HIGH"],
    "Table_Smoothness_SPARC.xlsx",
    expected_repetitions=2,
    decimals=3,
)



Saved: Statistical_Output\Table_Smoothness_SPARC.xlsx


,Metric,Participants,LOW,HIGH,Primary Test,Significance
0,Position,16,"-2.377 ± 0.046\n[-2.401, -2.352]","-2.388 ± 0.045\n[-2.411, -2.364]",Wilcoxon signed-rank,W = 40.000; p = .159; q = .159; r_rb = -0.412
1,Velocity,16,"-2.043 ± 0.603\n[-2.365, -1.721]","-1.824 ± 0.280\n[-1.973, -1.675]",Wilcoxon signed-rank,W = 34.000; p = .083; q = .133; r_rb = 0.500
2,Acceleration,16,"-2.753 ± 0.551\n[-3.046, -2.459]","-2.538 ± 0.321\n[-2.709, -2.367]",Paired-samples t-test,t(15) = 1.822; p = .088; q = .133; dz = 0.456


(         Metric  Participants                               LOW  \
 0      Position            16  -2.377 ± 0.046\n[-2.401, -2.352]   
 1      Velocity            16  -2.043 ± 0.603\n[-2.365, -1.721]   
 2  Acceleration            16  -2.753 ± 0.551\n[-3.046, -2.459]   
 
                                HIGH           Primary Test  \
 0  -2.388 ± 0.045\n[-2.411, -2.364]   Wilcoxon signed-rank   
 1  -1.824 ± 0.280\n[-1.973, -1.675]   Wilcoxon signed-rank   
 2  -2.538 ± 0.321\n[-2.709, -2.367]  Paired-samples t-test   
 
                                     Significance  
 0  W = 40.000; p = .159; q = .159; r_rb = -0.412  
 1   W = 34.000; p = .083; q = .133; r_rb = 0.500  
 2  t(15) = 1.822; p = .088; q = .133; dz = 0.456  ,
          Metric  Participants                               LOW  \
 0      Position            16  -2.377 ± 0.046\n[-2.401, -2.352]   
 1      Velocity            16  -2.043 ± 0.603\n[-2.365, -1.721]   
 2  Acceleration            16  -2.753 ± 0.551\n[-3.046, -2

## Table: Pelvis–Foot Redistribution

In [10]:
# =====================================================
# CONFIGURATION
# =====================================================
fs = 60  # Hz

participants = [f"P{i:03d}" for i in range(1, 17)]

conditions = {
    "LOW": [3, 4],
    "HIGH": [5, 6],
}


# =====================================================
# TIMING DICTIONARIES, SECONDS
# =====================================================
cut_times = {
    "P001": [12.55, 15.17, 6.90, 7.56],
    "P002": [15.81, 15.86, 7.40, 6.00],
}

reaction_times = {
    "P003": {
        "start": [10.28, 13.53, 7.50, 8.15],
        "end": [21.50, 23.80, 13.50, 12.50],
    },
    "P004": {
        "start": [8.31, 12.50, 5.88, 5.18],
        "end": [18.00, 22.00, 11.40, 9.80],
    },
    "P005": {
        "start": [12.16, 11.78, 5.58, 6.03],
        "end": [20.60, 20.80, 10.00, 10.50],
    },
    "P006": {
        "start": [10.66, 12.90, 5.81, 6.25],
        "end": [19.70, 20.50, 7.20, 9.80],
    },
    "P007": {
        "start": [10.00, 12.00, 6.10, 6.00],
        "end": [23.70, 15.60, 8.00, 9.60],
    },
    "P008": {
        "start": [11.00, 12.00, 6.00, 7.00],
        "end": [19.60, 24.00, 11.80, 10.20],
    },
    "P009": {
        "start": [10.29, 10.56, 5.86, 6.02],
        "end": [20.50, 23.00, 9.00, 12.00],
    },
    "P010": {
        "start": [12.00, 12.00, 6.00, 16.50],
        "end": [15.50, 16.20, 11.00, 20.00],
    },
    "P011": {
        "start": [10.94, 12.76, 6.10, 7.91],
        "end": [20.80, 24.00, 9.30, 14.00],
    },
    "P012": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.30, 17.00, 11.00, 10.50],
    },
    "P013": {
        "start": [10.91, 12.03, 6.41, 5.03],
        "end": [20.00, 19.80, 11.00, 10.00],
    },
    "P014": {
        "start": [16.00, 12.00, 6.00, 17.00],
        "end": [20.20, 19.00, 12.00, 24.00],
    },
    "P015": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.20, 17.50, 11.00, 11.00],
    },
    "P016": {
        "start": [12.00, 15.00, 6.00, 6.00],
        "end": [17.06, 22.00, 11.00, 12.00],
    },
}


# =====================================================
# ANALYSIS SETTINGS
# =====================================================
# The transient active-guidance phase analyzed in the paper.
PHASES_TO_RUN = ["P2"]

# Two prespecified pelvis-to-foot redistribution outcomes.
DISTAL_GROUPS_TO_RUN = [
    "Right_Foot",
    "Left_Foot",
]

VARIABLE_TO_RUN = "Redistribution_Index"


# =====================================================
# SEGMENT GROUPS
# =====================================================
distal_groups = {
    "Bilateral_UpperLeg": [
        "Right Upper Leg",
        "Left Upper Leg",
    ],
    "Bilateral_LowerLeg": [
        "Right Lower Leg",
        "Left Lower Leg",
    ],
    "Bilateral_Foot": [
        "Right Foot",
        "Left Foot",
    ],
    "Right_UpperLeg": ["Right Upper Leg"],
    "Left_UpperLeg": ["Left Upper Leg"],
    "Right_LowerLeg": ["Right Lower Leg"],
    "Left_LowerLeg": ["Left Lower Leg"],
    "Right_Foot": ["Right Foot"],
    "Left_Foot": ["Left Foot"],
}


# =====================================================
# CALCULATION FUNCTIONS
# =====================================================
def acceleration_magnitude(data, segment_name):
    return np.sqrt(
        data[f"{segment_name} x"] ** 2
        + data[f"{segment_name} y"] ** 2
        + data[f"{segment_name} z"] ** 2
    )


def mean_acceleration(data, segment_name):
    values = acceleration_magnitude(
        data,
        segment_name,
    )

    return values.replace(
        [np.inf, -np.inf],
        np.nan,
    ).mean()


# =====================================================
# REPORTING FUNCTIONS
# =====================================================
def format_p(p):
    if pd.isna(p):
        return "NA"

    if p < 0.001:
        return "< .001"

    return f"= {p:.3f}".replace("0.", ".")


def mean_sd_ci(values):
    """
    Return mean ± SD and 95% CI of the condition mean.
    """
    values = pd.Series(
        values,
        dtype=float,
    ).dropna()

    n = len(values)

    if n < 2:
        return "NA"

    mean = values.mean()
    sd = values.std(ddof=1)
    se = sd / np.sqrt(n)

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_low = mean - t_critical * se
    ci_high = mean + t_critical * se

    return (
        f"{mean:.3f} ± {sd:.3f}\n"
        f"[{ci_low:.3f}, {ci_high:.3f}]"
    )


def cohen_dz(differences):
    """
    Cohen's dz for paired HIGH-minus-LOW differences.
    """
    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    if len(differences) < 2:
        return np.nan

    sd_difference = differences.std(ddof=1)

    if sd_difference == 0:
        return np.nan

    return differences.mean() / sd_difference


def matched_rank_biserial(differences):
    """
    Matched-pairs rank-biserial correlation.

    Positive values indicate larger values under HIGH.
    """
    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    differences = differences[
        differences != 0
    ]

    if differences.empty:
        return np.nan

    ranks = stats.rankdata(
        np.abs(differences)
    )

    positive_rank_sum = ranks[
        differences.to_numpy() > 0
    ].sum()

    negative_rank_sum = ranks[
        differences.to_numpy() < 0
    ].sum()

    total_rank_sum = (
        positive_rank_sum
        + negative_rank_sum
    )

    if total_rank_sum == 0:
        return np.nan

    return (
        positive_rank_sum
        - negative_rank_sum
    ) / total_rank_sum


# =====================================================
# CALCULATE TRIAL-LEVEL REDISTRIBUTION INDICES
# =====================================================
trial_rows = []

for participant in participants:

    for condition, trials in conditions.items():

        for trial in trials:

            file_name = (
                f"{participant}-{trial:03d}.xlsx"
            )

            if not os.path.exists(file_name):
                continue

            try:
                trial_data = pd.read_excel(
                    file_name,
                    sheet_name="Segment Acceleration",
                )

                trial_data.columns = (
                    trial_data.columns
                    .astype(str)
                    .str.strip()
                )

                timing_index = trial - 3

                if participant in cut_times:
                    start_p2 = int(round(
                        cut_times[participant][timing_index]
                        * fs
                    ))
                    end_p2 = len(trial_data)
                    start_p3 = None
                else:
                    start_p2 = int(round(
                        reaction_times[participant]["start"][
                            timing_index
                        ] * fs
                    ))
                    end_p2 = int(round(
                        reaction_times[participant]["end"][
                            timing_index
                        ] * fs
                    ))
                    start_p3 = end_p2

                start_p2 = max(
                    0,
                    start_p2,
                )
                end_p2 = min(
                    len(trial_data),
                    end_p2,
                )

                if end_p2 <= start_p2:
                    continue

                phase_segments = {
                    "P2": trial_data.iloc[
                        start_p2:end_p2
                    ],
                }

                if start_p3 is not None:
                    phase_segments["P3"] = (
                        trial_data.iloc[start_p3:]
                    )

                for phase, segment_data in (
                    phase_segments.items()
                ):

                    if segment_data.empty:
                        continue

                    pelvis_mean = mean_acceleration(
                        segment_data,
                        "Pelvis",
                    )

                    for (
                        distal_group,
                        segment_names,
                    ) in distal_groups.items():

                        distal_values = [
                            mean_acceleration(
                                segment_data,
                                segment_name,
                            )
                            for segment_name
                            in segment_names
                        ]

                        distal_mean = np.nanmean(
                            distal_values
                        )

                        if (
                            not np.isfinite(pelvis_mean)
                            or not np.isfinite(distal_mean)
                            or distal_mean == 0
                        ):
                            redistribution_index = np.nan
                        else:
                            redistribution_index = (
                                pelvis_mean
                                / distal_mean
                            )

                        trial_rows.append({
                            "Participant": participant,
                            "Condition": condition,
                            "Trial": trial,
                            "Phase": phase,
                            "Distal_Group": distal_group,
                            "Pelvis_Mean_Acc": pelvis_mean,
                            "Distal_Mean_Acc": distal_mean,
                            "Redistribution_Index":
                                redistribution_index,
                        })

            except Exception:
                continue


trial_level = pd.DataFrame(
    trial_rows
)

if trial_level.empty:
    raise ValueError(
        "No redistribution-index data were computed. "
        "Check the working directory and workbook structure."
    )


pf_long = trial_level[
    (trial_level["Phase"].isin(PHASES_TO_RUN))
    & (trial_level["Distal_Group"].isin(DISTAL_GROUPS_TO_RUN))
].copy()

pf_long["Metric"] = pf_long["Distal_Group"].map({
    "Right_Foot": "Redistribution Index (pelvis/right foot)",
    "Left_Foot": "Redistribution Index (pelvis/left foot)",
})
metric_order = [
    "Redistribution Index (pelvis/right foot)",
    "Redistribution Index (pelvis/left foot)",
]
pf_long["Metric"] = pd.Categorical(
    pf_long["Metric"], categories=metric_order, ordered=True
)
pf_long = pf_long.sort_values(
    ["Metric", "Participant", "Condition", "Trial"]
).reset_index(drop=True)

run_repeated_family(
    pf_long,
    "Metric", VARIABLE_TO_RUN, "Participant", "Condition",
    ["LOW", "HIGH"],
    "Table_Pelvis_Foot_Redistribution.xlsx",
    expected_repetitions=2,
    decimals=3,
)



Saved: Statistical_Output\Table_Pelvis_Foot_Redistribution.xlsx


,Metric,Participants,LOW,HIGH,Primary Test,Significance
0,Redistribution Index (pelvis/right foot),16,"0.807 ± 0.184\n[0.709, 0.905]","0.725 ± 0.236\n[0.599, 0.851]",Paired-samples t-test,t(15) = -1.775; p = .096; q = .096; dz = -0.444
1,Redistribution Index (pelvis/left foot),16,"0.841 ± 0.223\n[0.722, 0.960]","0.705 ± 0.229\n[0.583, 0.827]",Paired-samples t-test,t(15) = -4.626; p < .001; q < .001; dz = -1.156


(                                     Metric  Participants  \
 0  Redistribution Index (pelvis/right foot)            16   
 1   Redistribution Index (pelvis/left foot)            16   
 
                              LOW                           HIGH  \
 0  0.807 ± 0.184\n[0.709, 0.905]  0.725 ± 0.236\n[0.599, 0.851]   
 1  0.841 ± 0.223\n[0.722, 0.960]  0.705 ± 0.229\n[0.583, 0.827]   
 
             Primary Test                                     Significance  
 0  Paired-samples t-test  t(15) = -1.775; p = .096; q = .096; dz = -0.444  
 1  Paired-samples t-test  t(15) = -4.626; p < .001; q < .001; dz = -1.156  ,
                                      Metric  Participants  \
 0  Redistribution Index (pelvis/right foot)            16   
 1   Redistribution Index (pelvis/left foot)            16   
 
                              LOW                           HIGH  \
 0  0.807 ± 0.184\n[0.709, 0.905]  0.725 ± 0.236\n[0.599, 0.851]   
 1  0.841 ± 0.223\n[0.722, 0.960]  0.705 ± 0.229\n

## SPM: Pelvis–Foot Redistribution Trajectories

In [12]:
# Input parameters and timing windows
fs = 60
participants = [f"P{i:03d}" for i in range(1, 17)]
trial_map = {3: "LOW", 4: "LOW", 5: "HIGH", 6: "HIGH"}
target_trials = [3, 4, 5, 6]
sheet_name = "Segment Acceleration"
n_points = 101
spm_outcomes = ["Pelvis/Left Foot Redistribution", "Pelvis/Right Foot Redistribution"]
spm_alpha_family = 0.05
spm_alpha_per_outcome = spm_alpha_family / len(spm_outcomes)  # Bonferroni across the two prespecified SPM outcomes

cut_times = {
    "P001": [12.55, 15.17, 6.90, 7.56],
    "P002": [15.81, 15.86, 7.40, 6.00],
}
reaction_times = {
    "P003": {"start": [10.28, 13.53, 7.50, 8.15], "end": [21.50, 23.80, 13.50, 12.50]},
    "P004": {"start": [8.31, 12.50, 5.88, 5.18], "end": [18.00, 22.00, 11.40, 9.80]},
    "P005": {"start": [12.16, 11.78, 5.58, 6.03], "end": [20.60, 20.80, 10.00, 10.50]},
    "P006": {"start": [10.66, 12.90, 5.81, 6.25], "end": [19.70, 20.50, 7.20, 9.80]},
    "P007": {"start": [10.00, 12.00, 6.10, 6.00], "end": [23.70, 15.60, 8.00, 9.60]},
    "P008": {"start": [11.00, 12.00, 6.00, 7.00], "end": [19.60, 24.00, 11.80, 10.20]},
    "P009": {"start": [10.29, 10.56, 5.86, 6.02], "end": [20.50, 23.00, 9.00, 12.00]},
    "P010": {"start": [12.00, 12.00, 6.00, 16.50], "end": [15.50, 16.20, 11.00, 20.00]},
    "P011": {"start": [10.94, 12.76, 6.10, 7.91], "end": [20.80, 24.00, 9.30, 14.00]},
    "P012": {"start": [12.00, 12.00, 6.00, 6.00], "end": [17.30, 17.00, 11.00, 10.50]},
    "P013": {"start": [10.91, 12.03, 6.41, 5.03], "end": [20.00, 19.80, 11.00, 10.00]},
    "P014": {"start": [16.00, 12.00, 6.00, 17.00], "end": [20.20, 19.00, 12.00, 24.00]},
    "P015": {"start": [12.00, 12.00, 6.00, 6.00], "end": [17.20, 17.50, 11.00, 11.00]},
    "P016": {"start": [12.00, 15.00, 6.00, 6.00], "end": [17.06, 22.00, 11.00, 12.00]},
}


def _clean_columns(df):
    df = df.copy(); df.columns = df.columns.astype(str).str.strip(); return df


def _time_normalize(signal, n_points=101):
    signal = np.asarray(signal, dtype=float)
    if signal.ndim != 1 or len(signal) < 3:
        return np.full(n_points, np.nan)
    valid = np.isfinite(signal)
    if valid.sum() < 3:
        return np.full(n_points, np.nan)
    if not valid.all():
        idx = np.arange(len(signal), dtype=float)
        signal = np.interp(idx, idx[valid], signal[valid])
    return interp1d(np.linspace(0, 100, len(signal)), signal, kind="linear", bounds_error=False,
                    fill_value="extrapolate")(np.linspace(0, 100, n_points))


def _magnitude(df, prefix):
    cols = [f"{prefix} x", f"{prefix} y", f"{prefix} z"]
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing columns: {missing}")
    x, y, z = [pd.to_numeric(df[c], errors="coerce").to_numpy(float) for c in cols]
    return np.sqrt(x*x + y*y + z*z)


rows, processing_log = [], []
for participant in participants:
    for trial in target_trials:
        file_name = f"{participant}-{trial:03d}.xlsx"
        if not os.path.exists(file_name):
            processing_log.append({"Participant": participant, "Trial": trial, "File": file_name, "Status": "Missing file"})
            continue
        try:
            td = _clean_columns(pd.read_excel(file_name, sheet_name=sheet_name))
            idx = trial - 3
            if participant in cut_times:
                start_time = cut_times[participant][idx]
                start_frame, end_frame = int(round(start_time * fs)), len(td)
                end_time = end_frame / fs
            else:
                start_time = reaction_times[participant]["start"][idx]
                end_time = reaction_times[participant]["end"][idx]
                start_frame, end_frame = int(round(start_time * fs)), int(round(end_time * fs))
            start_frame, end_frame = max(0, start_frame), min(len(td), end_frame)
            if end_frame <= start_frame:
                processing_log.append({"Participant": participant, "Trial": trial, "File": file_name, "Status": "Invalid cut window"})
                continue
            cropped = td.iloc[start_frame:end_frame].copy()
            pelvis = _magnitude(cropped, "Pelvis")
            left = _magnitude(cropped, "Left Foot")
            right = _magnitude(cropped, "Right Foot")
            eps = 1e-8
            rows.append({
                "Participant": participant, "Trial": trial, "Condition": trial_map[trial],
                "Start_Time_s": start_time, "End_Time_s": end_time, "Start_Frame": start_frame,
                "End_Frame": end_frame, "Segment_Samples": len(cropped),
                "Pelvis/Left Foot Redistribution": _time_normalize(pelvis / (pelvis + left + eps), n_points),
                "Pelvis/Right Foot Redistribution": _time_normalize(pelvis / (pelvis + right + eps), n_points),
            })
            processing_log.append({"Participant": participant, "Trial": trial, "File": file_name, "Status": "Processed"})
        except Exception as exc:
            processing_log.append({"Participant": participant, "Trial": trial, "File": file_name, "Status": f"{type(exc).__name__}: {exc}"})

curve_df = pd.DataFrame(rows)
processing_log_df = pd.DataFrame(processing_log)
if curve_df.empty:
    raise ValueError("No SPM trials were processed.")

participant_rows = []
for participant in participants:
    for condition in ["LOW", "HIGH"]:
        subset = curve_df[
            (curve_df["Participant"] == participant)
            & (curve_df["Condition"] == condition)
        ].copy()

        if subset.empty:
            continue

        row = {
            "Participant": participant,
            "Condition": condition,
            "Valid_Trials": len(subset),
        }

        for outcome in spm_outcomes:
            valid_curves = []
            for curve in subset[outcome].to_numpy():
                curve = np.asarray(curve, dtype=float)
                if curve.shape == (n_points,) and np.all(np.isfinite(curve)):
                    valid_curves.append(curve)

            if len(valid_curves) >= 1:
                row[outcome] = np.mean(np.vstack(valid_curves), axis=0)
            else:
                row[outcome] = np.full(n_points, np.nan)

        participant_rows.append(row)

participant_df = pd.DataFrame(participant_rows)

cluster_rows, summary_rows = [], []
for outcome in spm_outcomes:
    low_rows, high_rows, included = [], [], []
    for participant in participants:
        low = participant_df[(participant_df["Participant"] == participant) & (participant_df["Condition"] == "LOW")]
        high = participant_df[(participant_df["Participant"] == participant) & (participant_df["Condition"] == "HIGH")]
        if len(low) == 1 and len(high) == 1:
            l = np.asarray(low.iloc[0][outcome], float); h = np.asarray(high.iloc[0][outcome], float)
            if l.shape == (n_points,) and h.shape == (n_points,) and np.all(np.isfinite(l)) and np.all(np.isfinite(h)):
                low_rows.append(l); high_rows.append(h); included.append(participant)
    if len(included) < 3:
        summary_rows.append({"Outcome": outcome, "Participants": len(included), "df": max(len(included)-1, 0),
                             "Alpha family": spm_alpha_family, "Alpha used": spm_alpha_per_outcome,
                             "Significant clusters": 0})
        continue
    low_array, high_array = np.vstack(low_rows), np.vstack(high_rows)
    spm = spm1d.stats.ttest_paired(high_array, low_array)
    ti = spm.inference(alpha=spm_alpha_per_outcome, two_tailed=True, interp=True)
    clusters = list(getattr(ti, "clusters", []))
    summary_rows.append({"Outcome": outcome, "Participants": len(included), "df": len(included)-1,
                         "Alpha family": spm_alpha_family, "Alpha used": spm_alpha_per_outcome,
                         "SPM critical threshold": float(ti.zstar), "Significant clusters": len(clusters)})
    z = np.asarray(ti.z, float)
    for k, cluster in enumerate(clusters, start=1):
        endpoints = np.asarray(cluster.endpoints, float)
        start_node, end_node = float(endpoints[0]), float(endpoints[1])
        lo, hi = max(0, int(np.floor(start_node))), min(n_points-1, int(np.ceil(end_node)))
        inds = np.arange(lo, hi+1)
        peak_idx = int(inds[np.argmax(np.abs(z[inds]))])
        peak_t = float(z[peak_idx])
        cluster_rows.append({
            "Outcome": outcome, "Cluster": k, "Start_Percent": start_node * 100/(n_points-1),
            "End_Percent": end_node * 100/(n_points-1), "Cluster p": float(cluster.P),
            "Direction": "HIGH > LOW" if peak_t > 0 else "HIGH < LOW",
            "Peak SPM t": peak_t, "Peak Percent": peak_idx * 100/(n_points-1),
            "Participants": len(included), "df": len(included)-1,
        })

spm_summary = pd.DataFrame(summary_rows)
spm_clusters = pd.DataFrame(cluster_rows)
participant_export = participant_df[["Participant", "Condition", "Valid_Trials"]].copy()
metadata = pd.DataFrame({
    "Item": ["Experimental unit", "Repeated-trial handling", "SPM test", "Multiple-comparison correction"],
    "Specification": [
        "Participant",
        "Available trials are averaged within participant and condition before SPM inference; if only one valid trial is available, that trial is used.",
        "Two-tailed participant-level paired SPM1D t-test; df = number of paired participants - 1.",
        "SPM1D controls the continuum-wise error within each trajectory; Bonferroni alpha = .05/2 = .025 is used across the two prespecified pelvis-foot trajectory outcomes.",
    ],
})
spm_path = OUTPUT_DIR / "SPM_Pelvis_Foot_Redistribution.xlsx"
with pd.ExcelWriter(spm_path, engine="openpyxl") as writer:
    spm_summary.to_excel(writer, sheet_name="SPM Summary", index=False)
    spm_clusters.to_excel(writer, sheet_name="Significant Clusters", index=False)
    participant_export.to_excel(writer, sheet_name="Participant Means", index=False)
    curve_df.drop(columns=spm_outcomes).to_excel(writer, sheet_name="Trial Processing", index=False)
    processing_log_df.to_excel(writer, sheet_name="Processing Log", index=False)
    metadata.to_excel(writer, sheet_name="Analysis Metadata", index=False)
_format_workbook(spm_path)


## Table: EMG Redistribution

In [14]:
# =====================================================
# LOAD DATASET
# =====================================================
data = pd.read_excel(
    "EMG Data Analysis.xlsx",
    sheet_name="TransientState",
).copy()

data.columns = (
    data.columns
    .astype(str)
    .str.strip()
)

# Restrict analysis to Area 1.
data = data[
    data["Area"] == 1
].copy()


# =====================================================
# CONDITION MAPPING
# =====================================================
condition_map = {
    "WRLS01": "LOW",
    "WRLS02": "LOW",
    "WRHS01": "HIGH",
    "WRHS02": "HIGH",
}

data["Group"] = data[
    "Condition"
].map(condition_map)

data = data[
    data["Group"].isin(
        ["LOW", "HIGH"]
    )
].copy()


# =====================================================
# MUSCLE GROUPS
# =====================================================
right_proximal = ["RRF"]
right_distal = ["RTA", "RGAL"]

left_proximal = ["LRF"]
left_distal = ["LTA", "LGAL"]


# =====================================================
# REPORTING FUNCTIONS
# =====================================================
def format_p(p):
    if pd.isna(p):
        return "NA"

    if p < 0.001:
        return "< .001"

    return f"= {p:.3f}".replace("0.", ".")


def mean_sd_ci(values):
    """
    Return mean ± SD and 95% CI of the condition mean.
    """
    values = pd.Series(
        values,
        dtype=float,
    ).dropna()

    n = len(values)

    if n < 2:
        return "NA"

    mean = values.mean()
    sd = values.std(ddof=1)
    se = sd / np.sqrt(n)

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_low = mean - t_critical * se
    ci_high = mean + t_critical * se

    return (
        f"{mean:.3f} ± {sd:.3f}\n"
        f"[{ci_low:.3f}, {ci_high:.3f}]"
    )


def cohen_dz(differences):
    """
    Cohen's dz for paired HIGH-minus-LOW differences.
    """
    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    if len(differences) < 2:
        return np.nan

    sd_difference = differences.std(ddof=1)

    if sd_difference == 0:
        return np.nan

    return (
        differences.mean()
        / sd_difference
    )


def matched_rank_biserial(differences):
    """
    Matched-pairs rank-biserial correlation.

    Positive values indicate larger values under HIGH.
    """
    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    differences = differences[
        differences != 0
    ]

    if differences.empty:
        return np.nan

    ranks = stats.rankdata(
        np.abs(differences)
    )

    positive_rank_sum = ranks[
        differences.to_numpy() > 0
    ].sum()

    negative_rank_sum = ranks[
        differences.to_numpy() < 0
    ].sum()

    total_rank_sum = (
        positive_rank_sum
        + negative_rank_sum
    )

    if total_rank_sum == 0:
        return np.nan

    return (
        positive_rank_sum
        - negative_rank_sum
    ) / total_rank_sum


# =====================================================
# COMPUTE REDISTRIBUTION RATIO PER ORIGINAL TRIAL
# =====================================================
# Each Condition code represents one original trial:
# WRLS01 and WRLS02 = two LOW trials
# WRHS01 and WRHS02 = two HIGH trials

trial_rows = []

group_columns = [
    "Participant",
    "Condition",
    "Group",
]

for keys, trial_data in data.groupby(
    group_columns
):

    participant = keys[0]
    condition = keys[1]
    group = keys[2]

    # Require all specified sensors to be represented.
    available_sensors = set(
        trial_data["sensor"].dropna()
    )

    required_sensors = set(
        right_proximal
        + right_distal
        + left_proximal
        + left_distal
    )

    if not required_sensors.issubset(
        available_sensors
    ):
        continue

    right_proximal_sum = trial_data[
        trial_data["sensor"].isin(
            right_proximal
        )
    ]["iEMG"].sum(min_count=1)

    right_distal_sum = trial_data[
        trial_data["sensor"].isin(
            right_distal
        )
    ]["iEMG"].sum(min_count=1)

    left_proximal_sum = trial_data[
        trial_data["sensor"].isin(
            left_proximal
        )
    ]["iEMG"].sum(min_count=1)

    left_distal_sum = trial_data[
        trial_data["sensor"].isin(
            left_distal
        )
    ]["iEMG"].sum(min_count=1)

    right_denominator = (
        right_proximal_sum
        + right_distal_sum
    )

    left_denominator = (
        left_proximal_sum
        + left_distal_sum
    )

    if (
        not np.isfinite(right_denominator)
        or not np.isfinite(left_denominator)
        or right_denominator == 0
        or left_denominator == 0
    ):
        continue

    right_ratio = (
        right_proximal_sum
        / right_denominator
    )

    left_ratio = (
        left_proximal_sum
        / left_denominator
    )

    trial_rows.append({
        "Participant": participant,
        "Condition": condition,
        "Group": group,
        "RR_Right": right_ratio,
        "RR_Left": left_ratio,
    })


trial_level = pd.DataFrame(
    trial_rows
)

if trial_level.empty:
    raise ValueError(
        "No EMG redistribution ratios were computed. "
        "Check the input workbook and sensor labels."
    )


emgr_long = trial_level.melt(
    id_vars=["Participant", "Condition", "Group"],
    value_vars=["RR_Right", "RR_Left"],
    var_name="Metric",
    value_name="Value",
)
emgr_long["Metric"] = emgr_long["Metric"].map({
    "RR_Right": "Right EMG Redistribution",
    "RR_Left": "Left EMG Redistribution",
})
metric_order = ["Right EMG Redistribution", "Left EMG Redistribution"]
emgr_long["Metric"] = pd.Categorical(
    emgr_long["Metric"], categories=metric_order, ordered=True
)
emgr_long = emgr_long.sort_values(
    ["Metric", "Participant", "Group", "Condition"]
).reset_index(drop=True)

run_repeated_family(
    emgr_long,
    "Metric", "Value", "Participant", "Group",
    ["LOW", "HIGH"],
    "Table_EMG_Redistribution.xlsx",
    expected_repetitions=2,
    decimals=3,
)



Saved: Statistical_Output\Table_EMG_Redistribution.xlsx


,Metric,Participants,LOW,HIGH,Primary Test,Significance
0,Right EMG Redistribution,15,"0.232 ± 0.074\n[0.191, 0.273]","0.239 ± 0.084\n[0.192, 0.285]",Paired-samples t-test,t(14) = 0.457; p = .655; q = .655; dz = 0.118
1,Left EMG Redistribution,15,"0.296 ± 0.203\n[0.183, 0.408]","0.271 ± 0.203\n[0.158, 0.384]",Wilcoxon signed-rank,W = 42.000; p = .330; q = .655; r_rb = -0.300


(                     Metric  Participants                            LOW  \
 0  Right EMG Redistribution            15  0.232 ± 0.074\n[0.191, 0.273]   
 1   Left EMG Redistribution            15  0.296 ± 0.203\n[0.183, 0.408]   
 
                             HIGH           Primary Test  \
 0  0.239 ± 0.084\n[0.192, 0.285]  Paired-samples t-test   
 1  0.271 ± 0.203\n[0.158, 0.384]   Wilcoxon signed-rank   
 
                                     Significance  
 0  t(14) = 0.457; p = .655; q = .655; dz = 0.118  
 1  W = 42.000; p = .330; q = .655; r_rb = -0.300  ,
                      Metric  Participants                            LOW  \
 0  Right EMG Redistribution            15  0.232 ± 0.074\n[0.191, 0.273]   
 1   Left EMG Redistribution            15  0.296 ± 0.203\n[0.183, 0.408]   
 
                             HIGH           Primary Test    Contrast  \
 0  0.239 ± 0.084\n[0.192, 0.285]  Paired-samples t-test  HIGH − LOW   
 1  0.271 ± 0.203\n[0.158, 0.384]   Wilcoxon signed

## Table: Margin of Stability (MoS)

In [16]:
fs = 60
g = 9.81

participants = [f"P{i:03d}" for i in range(1, 17)]

conditions = {
    "LOW": [3, 4],
    "HIGH": [5, 6],
}


# =====================================================
# TIMING DICTIONARIES
# =====================================================
cut_times = {
    "P001": [12.55, 15.17, 6.90, 7.56],
    "P002": [15.81, 15.86, 7.40, 6.00],
}

reaction_times = {
    "P003": {
        "start": [10.28, 13.53, 7.50, 8.15],
        "end": [21.50, 23.80, 13.50, 12.50],
    },
    "P004": {
        "start": [8.31, 12.50, 5.88, 5.18],
        "end": [18.00, 22.00, 11.40, 9.80],
    },
    "P005": {
        "start": [12.16, 11.78, 5.58, 6.03],
        "end": [20.60, 20.80, 10.00, 10.50],
    },
    "P006": {
        "start": [10.66, 12.90, 5.81, 6.25],
        "end": [19.70, 20.50, 7.20, 9.80],
    },
    "P007": {
        "start": [10.00, 12.00, 6.10, 6.00],
        "end": [23.70, 15.60, 8.00, 9.60],
    },
    "P008": {
        "start": [11.00, 12.00, 6.00, 7.00],
        "end": [19.60, 24.00, 11.80, 10.20],
    },
    "P009": {
        "start": [10.29, 10.56, 5.86, 6.02],
        "end": [20.50, 23.00, 9.00, 12.00],
    },
    "P010": {
        "start": [12.00, 12.00, 6.00, 16.50],
        "end": [15.50, 16.20, 11.00, 20.00],
    },
    "P011": {
        "start": [10.94, 12.76, 6.10, 7.91],
        "end": [20.80, 24.00, 9.30, 14.00],
    },
    "P012": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.30, 17.00, 11.00, 10.50],
    },
    "P013": {
        "start": [10.91, 12.03, 6.41, 5.03],
        "end": [20.00, 19.80, 11.00, 10.00],
    },
    "P014": {
        "start": [16.00, 12.00, 6.00, 17.00],
        "end": [20.20, 19.00, 12.00, 24.00],
    },
    "P015": {
        "start": [12.00, 12.00, 6.00, 6.00],
        "end": [17.20, 17.50, 11.00, 11.00],
    },
    "P016": {
        "start": [12.00, 15.00, 6.00, 6.00],
        "end": [17.06, 22.00, 11.00, 12.00],
    },
}


# =====================================================
# REPORTING FUNCTIONS
# =====================================================
def format_p(p):
    if pd.isna(p):
        return "NA"

    if p < 0.001:
        return "< .001"

    return f"= {p:.3f}".replace("0.", ".")


def mean_sd_ci(values):
    """
    Return mean ± SD and 95% CI of the condition mean.
    """
    values = pd.Series(
        values,
        dtype=float,
    ).dropna()

    n = len(values)

    if n < 2:
        return "NA"

    mean_value = values.mean()
    sd_value = values.std(ddof=1)
    se_value = sd_value / np.sqrt(n)

    t_critical = stats.t.ppf(
        0.975,
        df=n - 1,
    )

    ci_low = mean_value - t_critical * se_value
    ci_high = mean_value + t_critical * se_value

    return (
        f"{mean_value:.3f} ± {sd_value:.3f}\n"
        f"[{ci_low:.3f}, {ci_high:.3f}]"
    )


def cohen_dz(differences):
    """
    Cohen's dz for paired HIGH-minus-LOW differences.
    """
    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    if len(differences) < 2:
        return np.nan

    sd_difference = differences.std(ddof=1)

    if sd_difference == 0:
        return np.nan

    return (
        differences.mean()
        / sd_difference
    )


def matched_rank_biserial(differences):
    """
    Matched-pairs rank-biserial correlation.

    Positive values indicate larger values under HIGH.
    """
    differences = pd.Series(
        differences,
        dtype=float,
    ).dropna()

    differences = differences[
        differences != 0
    ]

    if differences.empty:
        return np.nan

    ranks = stats.rankdata(
        np.abs(differences)
    )

    positive_rank_sum = ranks[
        differences.to_numpy() > 0
    ].sum()

    negative_rank_sum = ranks[
        differences.to_numpy() < 0
    ].sum()

    total_rank_sum = (
        positive_rank_sum
        + negative_rank_sum
    )

    if total_rank_sum == 0:
        return np.nan

    return (
        positive_rank_sum
        - negative_rank_sum
    ) / total_rank_sum


# =====================================================
# COMPUTE TRIAL-LEVEL ML MARGIN OF STABILITY
# =====================================================
trial_rows = []

required_com_columns = [
    "CoM pos y",
    "CoM vel y",
    "CoM pos z",
]

required_position_columns = [
    "Left Foot y",
    "Right Foot y",
]

for participant in participants:

    for condition, trials in conditions.items():

        for trial in trials:

            file_name = (
                f"{participant}-{trial:03d}.xlsx"
            )

            if not os.path.exists(file_name):
                continue

            try:
                com = pd.read_excel(
                    file_name,
                    sheet_name="Center of Mass",
                )

                segment_position = pd.read_excel(
                    file_name,
                    sheet_name="Segment Position",
                )

                com.columns = (
                    com.columns
                    .astype(str)
                    .str.strip()
                )

                segment_position.columns = (
                    segment_position.columns
                    .astype(str)
                    .str.strip()
                )

                if any(
                    column not in com.columns
                    for column in required_com_columns
                ):
                    continue

                if any(
                    column not in segment_position.columns
                    for column in required_position_columns
                ):
                    continue

                timing_index = trial - 3

                if participant in cut_times:
                    start_frame = int(round(
                        cut_times[participant][timing_index]
                        * fs
                    ))
                    end_frame = len(com)

                else:
                    start_frame = int(round(
                        reaction_times[participant]["start"][
                            timing_index
                        ] * fs
                    ))

                    end_frame = int(round(
                        reaction_times[participant]["end"][
                            timing_index
                        ] * fs
                    ))

                start_frame = max(
                    0,
                    start_frame,
                )

                end_frame = min(
                    end_frame,
                    len(com),
                    len(segment_position),
                )

                if end_frame <= start_frame:
                    continue

                com_y = (
                    com["CoM pos y"]
                    .iloc[start_frame:end_frame]
                    .to_numpy(dtype=float)
                )

                com_velocity_y = (
                    com["CoM vel y"]
                    .iloc[start_frame:end_frame]
                    .to_numpy(dtype=float)
                )

                com_z = (
                    com["CoM pos z"]
                    .iloc[start_frame:end_frame]
                    .to_numpy(dtype=float)
                )

                left_foot_y = (
                    segment_position["Left Foot y"]
                    .iloc[start_frame:end_frame]
                    .to_numpy(dtype=float)
                )

                right_foot_y = (
                    segment_position["Right Foot y"]
                    .iloc[start_frame:end_frame]
                    .to_numpy(dtype=float)
                )

                valid_signal = (
                    np.isfinite(com_y)
                    & np.isfinite(com_velocity_y)
                    & np.isfinite(com_z)
                    & np.isfinite(left_foot_y)
                    & np.isfinite(right_foot_y)
                )

                if not np.any(valid_signal):
                    continue

                com_y = com_y[valid_signal]
                com_velocity_y = (
                    com_velocity_y[valid_signal]
                )
                com_z = com_z[valid_signal]
                left_foot_y = left_foot_y[valid_signal]
                right_foot_y = right_foot_y[valid_signal]

                pendulum_length = np.mean(com_z)

                if (
                    not np.isfinite(pendulum_length)
                    or pendulum_length <= 0
                ):
                    continue

                omega_0 = np.sqrt(
                    g / pendulum_length
                )

                xcom_ml = (
                    com_y
                    + com_velocity_y / omega_0
                )

                bos_ml = np.maximum(
                    left_foot_y,
                    right_foot_y,
                )

                mos_ml = bos_ml - xcom_ml
                mos_ml = mos_ml[
                    np.isfinite(mos_ml)
                ]

                if len(mos_ml) == 0:
                    continue

                trial_rows.append({
                    "Participant": participant,
                    "Condition": condition,
                    "Trial": trial,
                    "Mean_ML_MoS": np.mean(mos_ml),
                    "Min_ML_MoS": np.min(mos_ml),
                    "Max_ML_MoS": np.max(mos_ml),
                })

            except Exception:
                continue


trial_level = pd.DataFrame(
    trial_rows
)

if trial_level.empty:
    raise ValueError(
        "No MoS results were computed. Check the working "
        "directory and input workbook structure."
    )


mos_long = trial_level.melt(
    id_vars=["Participant", "Condition", "Trial"],
    value_vars=["Mean_ML_MoS", "Min_ML_MoS", "Max_ML_MoS"],
    var_name="Metric",
    value_name="Value",
)
mos_long["Metric"] = mos_long["Metric"].map({
    "Mean_ML_MoS": "Mean MoS",
    "Min_ML_MoS": "Minimum MoS",
    "Max_ML_MoS": "Maximum MoS",
})
metric_order = ["Mean MoS", "Minimum MoS", "Maximum MoS"]
mos_long["Metric"] = pd.Categorical(
    mos_long["Metric"], categories=metric_order, ordered=True
)
mos_long = mos_long.sort_values(
    ["Metric", "Participant", "Condition", "Trial"]
).reset_index(drop=True)

run_repeated_family(
    mos_long,
    "Metric", "Value", "Participant", "Condition",
    ["LOW", "HIGH"],
    "Table_Margin_of_Stability.xlsx",
    expected_repetitions=2,
    decimals=3,
)



Saved: Statistical_Output\Table_Margin_of_Stability.xlsx


,Metric,Participants,LOW,HIGH,Primary Test,Significance
0,Mean MoS,16,"0.062 ± 0.057\n[0.032, 0.092]","0.059 ± 0.058\n[0.028, 0.090]",Paired-samples t-test,t(15) = -0.370; p = .717; q = .717; dz = -0.092
1,Minimum MoS,16,"-0.024 ± 0.050\n[-0.051, 0.003]","-0.029 ± 0.040\n[-0.050, -0.007]",Paired-samples t-test,t(15) = -0.678; p = .508; q = .717; dz = -0.169
2,Maximum MoS,16,"0.345 ± 0.211\n[0.233, 0.458]","0.295 ± 0.215\n[0.181, 0.410]",Paired-samples t-test,t(15) = -2.507; p = .024; q = .073; dz = -0.627


(        Metric  Participants                              LOW  \
 0     Mean MoS            16    0.062 ± 0.057\n[0.032, 0.092]   
 1  Minimum MoS            16  -0.024 ± 0.050\n[-0.051, 0.003]   
 2  Maximum MoS            16    0.345 ± 0.211\n[0.233, 0.458]   
 
                                HIGH           Primary Test  \
 0     0.059 ± 0.058\n[0.028, 0.090]  Paired-samples t-test   
 1  -0.029 ± 0.040\n[-0.050, -0.007]  Paired-samples t-test   
 2     0.295 ± 0.215\n[0.181, 0.410]  Paired-samples t-test   
 
                                       Significance  
 0  t(15) = -0.370; p = .717; q = .717; dz = -0.092  
 1  t(15) = -0.678; p = .508; q = .717; dz = -0.169  
 2  t(15) = -2.507; p = .024; q = .073; dz = -0.627  ,
         Metric  Participants                              LOW  \
 0     Mean MoS            16    0.062 ± 0.057\n[0.032, 0.092]   
 1  Minimum MoS            16  -0.024 ± 0.050\n[-0.051, 0.003]   
 2  Maximum MoS            16    0.345 ± 0.211\n[0.233, 0.458]   

## Table: Steady-State Kinematics

In [18]:
input_path = "Xsens Data Analysis.xlsx"
sheet_name = "Sheet1"
area_value = 1
subject_col = "Participant"
trial_col = "Condition"

condition_order = ["No Robot", "Low Speed", "High Speed"]

condition_map = {
    "NR01": "No Robot", "NR02": "No Robot",
    "WRLS01": "Low Speed", "WRLS02": "Low Speed",
    "WRHS01": "High Speed", "WRHS02": "High Speed",
}

data = pd.read_excel(input_path, sheet_name=sheet_name).copy()
data.columns = data.columns.astype(str).str.strip()
data = data[data["Area"] == area_value].copy()
data["Group"] = data[trial_col].map(condition_map)
data = data[data["Group"].isin(condition_order)].copy()

data["CoM_pos_mag"] = np.sqrt(data["CoM pos x"]**2 + data["CoM pos y"]**2 + data["CoM pos z"]**2)
data["CoM_vel_mag"] = np.sqrt(data["CoM vel x"]**2 + data["CoM vel y"]**2 + data["CoM vel z"]**2)
data["CoM_acc_mag"] = np.sqrt(data["CoM acc x"]**2 + data["CoM acc y"]**2 + data["CoM acc z"]**2)

kinematic_columns = [
    "CoM_pos_mag",
    "CoM_vel_mag",
    "CoM_acc_mag",
    "Right Shoulder Flexion/Extension",
    "Right Shoulder Internal/External Rotation",
    "Right Shoulder Abduction/Adduction",
    "Right Knee Flexion/Extension",
    "Right Knee Internal/External Rotation",
    "Right Knee Abduction/Adduction",
    "Left Knee Flexion/Extension",
    "Left Knee Internal/External Rotation",
    "Left Knee Abduction/Adduction",
]

rename = {
    "CoM_pos_mag": "CoM Position Magnitude",
    "CoM_vel_mag": "CoM Velocity Magnitude",
    "CoM_acc_mag": "CoM Acceleration Magnitude",
}

kin_rows = []

for variable in kinematic_columns:
    if variable not in data.columns:
        continue

    # One value per participant x ORIGINAL trial.
    t = (
        data.groupby(
            [subject_col, trial_col, "Group"],
            as_index=False
        )[variable]
        .mean()
        .rename(columns={variable: "Value"})
    )
    t["Metric"] = rename.get(variable, variable)
    kin_rows.append(
        t[["Metric", subject_col, trial_col, "Group", "Value"]]
    )

kin_long = pd.concat(kin_rows, ignore_index=True)

# EXACT requested output sequence
metric_order = [
    "CoM Position Magnitude",
    "CoM Velocity Magnitude",
    "CoM Acceleration Magnitude",
    "Right Shoulder Flexion/Extension",
    "Right Shoulder Internal/External Rotation",
    "Right Shoulder Abduction/Adduction",
    "Right Knee Flexion/Extension",
    "Right Knee Internal/External Rotation",
    "Right Knee Abduction/Adduction",
    "Left Knee Flexion/Extension",
    "Left Knee Internal/External Rotation",
    "Left Knee Abduction/Adduction",
]
kin_long["Metric"] = pd.Categorical(
    kin_long["Metric"], categories=metric_order, ordered=True
)
kin_long = kin_long.sort_values(
    ["Metric", subject_col, "Group", trial_col]
).reset_index(drop=True)

run_repeated_family(
    kin_long,
    "Metric", "Value", subject_col, "Group",
    condition_order,
    "Table_Kinematics.xlsx",
    expected_repetitions=2,
    decimals=2,
)



Saved: Statistical_Output\Table_Kinematics.xlsx


,Metric,Participants,No Robot,Low Speed,High Speed,Primary Test,Significance
0,CoM Position Magnitude,16,"8.15 ± 2.39\n[6.88, 9.43]","8.22 ± 2.24\n[7.03, 9.41]","7.89 ± 2.38\n[6.62, 9.16]",Friedman test,χ²(2) = 4.500; p = .105; q = .115; Kendall's W...
1,CoM Velocity Magnitude,16,"0.04 ± 0.01\n[0.04, 0.05]","0.07 ± 0.04\n[0.05, 0.09]","0.07 ± 0.03\n[0.05, 0.09]",Friedman test,χ²(2) = 24.000; p < .001; q < .001; Kendall's ...
2,CoM Acceleration Magnitude,16,"0.19 ± 0.04\n[0.17, 0.22]","0.29 ± 0.11\n[0.23, 0.35]","0.28 ± 0.10\n[0.22, 0.33]",Friedman test,χ²(2) = 24.125; p < .001; q < .001; Kendall's ...
3,Right Shoulder Flexion/Extension,16,"35.89 ± 6.59\n[32.39, 39.40]","36.49 ± 7.67\n[32.40, 40.58]","39.57 ± 10.12\n[34.17, 44.96]",Repeated-measures ANOVA,"F(2.000, 30.000) = 2.236; p = .124; q = .124; ..."
4,Right Shoulder Internal/External Rotation,16,"19.23 ± 3.57\n[17.33, 21.13]","22.90 ± 4.31\n[20.60, 25.19]","24.50 ± 3.99\n[22.37, 26.63]",Repeated-measures ANOVA,"F(2.000, 30.000) = 13.700; p < .001; q < .001;..."
5,Right Shoulder Abduction/Adduction,16,"24.95 ± 5.17\n[22.20, 27.71]","28.08 ± 3.92\n[25.99, 30.16]","30.44 ± 3.74\n[28.45, 32.43]",Repeated-measures ANOVA,"F(2.000, 30.000) = 13.748; p < .001; q < .001;..."
6,Right Knee Flexion/Extension,16,"4.83 ± 2.47\n[3.52, 6.15]","6.39 ± 3.10\n[4.74, 8.04]","7.37 ± 3.77\n[5.36, 9.38]",Repeated-measures ANOVA,"F(2.000, 30.000) = 12.722; p < .001; q < .001;..."
7,Right Knee Internal/External Rotation,16,"1.73 ± 0.41\n[1.51, 1.95]","2.15 ± 0.92\n[1.66, 2.64]","2.12 ± 0.58\n[1.81, 2.43]",Friedman test,χ²(2) = 10.500; p = .005; q = .006; Kendall's ...
8,Right Knee Abduction/Adduction,16,"0.47 ± 0.39\n[0.26, 0.68]","0.62 ± 0.47\n[0.37, 0.88]","0.67 ± 0.47\n[0.42, 0.93]",Friedman test,χ²(2) = 18.875; p < .001; q < .001; Kendall's ...
9,Left Knee Flexion/Extension,16,"3.55 ± 1.56\n[2.72, 4.38]","5.69 ± 2.38\n[4.42, 6.96]","5.93 ± 2.78\n[4.45, 7.41]",Repeated-measures ANOVA,"F(2.000, 30.000) = 10.789; p < .001; q < .001;..."



Holm-adjusted post-hoc comparisons:


,Metric,Comparison,N,Test,Statistic,Raw p,Effect Size,Effect Size Type,Holm-adjusted p
0,CoM Velocity Magnitude,Low Speed − No Robot,16,Wilcoxon signed-rank,0.000000,0.000031,1.000000,Rank-biserial r,0.000092
1,CoM Velocity Magnitude,High Speed − No Robot,16,Wilcoxon signed-rank,0.000000,0.000031,1.000000,Rank-biserial r,0.000092
2,CoM Velocity Magnitude,High Speed − Low Speed,16,Wilcoxon signed-rank,68.000000,1.000000,0.000000,Rank-biserial r,1.000000
3,CoM Acceleration Magnitude,Low Speed − No Robot,16,Wilcoxon signed-rank,0.000000,0.000031,1.000000,Rank-biserial r,0.000092
4,CoM Acceleration Magnitude,High Speed − No Robot,16,Wilcoxon signed-rank,0.000000,0.000031,1.000000,Rank-biserial r,0.000092
5,CoM Acceleration Magnitude,High Speed − Low Speed,16,Wilcoxon signed-rank,65.000000,0.899933,0.044118,Rank-biserial r,0.899933
6,Right Shoulder Internal/External Rotation,Low Speed − No Robot,16,Paired-samples t-test,3.934315,0.001325,0.983579,Cohen dz,0.002650
7,Right Shoulder Internal/External Rotation,High Speed − No Robot,16,Paired-samples t-test,4.750442,0.000258,1.187610,Cohen dz,0.000773
8,Right Shoulder Internal/External Rotation,High Speed − Low Speed,16,Paired-samples t-test,1.528770,0.147133,0.382192,Cohen dz,0.147133
9,Right Shoulder Abduction/Adduction,Low Speed − No Robot,16,Paired-samples t-test,2.781476,0.013972,0.695369,Cohen dz,0.027944


(                                       Metric  Participants  \
 0                      CoM Position Magnitude            16   
 1                      CoM Velocity Magnitude            16   
 2                  CoM Acceleration Magnitude            16   
 3            Right Shoulder Flexion/Extension            16   
 4   Right Shoulder Internal/External Rotation            16   
 5          Right Shoulder Abduction/Adduction            16   
 6                Right Knee Flexion/Extension            16   
 7       Right Knee Internal/External Rotation            16   
 8              Right Knee Abduction/Adduction            16   
 9                 Left Knee Flexion/Extension            16   
 10       Left Knee Internal/External Rotation            16   
 11              Left Knee Abduction/Adduction            16   
 
                         No Robot                     Low Speed  \
 0      8.15 ± 2.39\n[6.88, 9.43]     8.22 ± 2.24\n[7.03, 9.41]   
 1      0.04 ± 0.01\n[0.04, 0.05

## Table: Steady-State EMG

In [20]:
input_path = "EMG Data Analysis.xlsx"
sheet_name = "SteadyState"
subject_col = "Participant"
condition_col = "Condition"
sensor_col = "sensor"
value_col = "iEMG"
area_col = "Area"
condition_order = ["NR", "LOW", "HIGH"]
condition_map = {
    "NR01": "NR", "NR02": "NR", "WRLS01": "LOW", "WRLS02": "LOW", "WRHS01": "HIGH", "WRHS02": "HIGH",
}
sensors_to_analyze = ["LGAL", "LRF", "LTA", "LUT", "RBB", "RGAL", "RRF", "RTA", "RTB", "RUT"]

data = pd.read_excel(input_path, sheet_name=sheet_name).copy()
data.columns = data.columns.astype(str).str.strip()
required = [subject_col, condition_col, sensor_col, value_col]
missing = [c for c in required if c not in data.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")
if area_col in data.columns:
    data = data[data[area_col] == 1].copy()
data[subject_col] = data[subject_col].astype(str).str.strip()
data[condition_col] = data[condition_col].astype(str).str.strip()
data[sensor_col] = data[sensor_col].astype(str).str.strip().str.upper()
data[value_col] = pd.to_numeric(data[value_col], errors="coerce")
data["Group"] = data[condition_col].map(condition_map)
data = data[data["Group"].isin(condition_order) & data[sensor_col].isin(sensors_to_analyze)].dropna(subset=[subject_col, condition_col, sensor_col, value_col, "Group"]).copy()
trial_level = (
    data.groupby(
        [subject_col, sensor_col, condition_col, "Group"],
        as_index=False
    )[value_col]
    .mean()
    .rename(columns={sensor_col: "Metric", value_col: "Value"})
)

# EXACT sensor order requested/defined above
trial_level["Metric"] = pd.Categorical(
    trial_level["Metric"],
    categories=sensors_to_analyze,
    ordered=True,
)
trial_level = trial_level.sort_values(
    ["Metric", subject_col, "Group", condition_col]
).reset_index(drop=True)

run_repeated_family(
    trial_level,
    "Metric", "Value", subject_col, "Group",
    condition_order,
    "Table_EMG_iEMG.xlsx",
    expected_repetitions=2,
    decimals=2,
)



Saved: Statistical_Output\Table_EMG_iEMG.xlsx


,Metric,Participants,NR,LOW,HIGH,Primary Test,Significance
0,LGAL,15,"25.91 ± 14.76\n[17.73, 34.08]","27.76 ± 16.30\n[18.73, 36.78]","24.34 ± 13.49\n[16.87, 31.81]",Repeated-measures ANOVA,"F(2.000, 28.000) = 2.113; p = .140; q = .200; ..."
1,LRF,15,"30.75 ± 41.10\n[7.99, 53.51]","27.35 ± 35.44\n[7.72, 46.98]","29.13 ± 39.57\n[7.21, 51.04]",Friedman test,χ²(2) = 1.733; p = .420; q = .467; Kendall's W...
2,LTA,15,"14.46 ± 12.00\n[7.81, 21.10]","15.12 ± 12.93\n[7.96, 22.28]","16.46 ± 13.64\n[8.91, 24.02]",Repeated-measures ANOVA,"F(2.000, 28.000) = 3.375; p = .049; q = .097; ..."
3,LUT,15,"68.67 ± 53.14\n[39.24, 98.10]","68.63 ± 51.08\n[40.34, 96.92]","71.50 ± 52.94\n[42.18, 100.81]",Repeated-measures ANOVA,"F(1.409, 19.722) = 0.457; GG-corrected; p = .5..."
4,RBB,15,"20.56 ± 12.52\n[13.63, 27.50]","20.15 ± 12.88\n[13.02, 27.29]","21.67 ± 14.52\n[13.63, 29.71]",Repeated-measures ANOVA,"F(2.000, 28.000) = 2.609; p = .091; q = .152; ..."
5,RGAL,15,"22.18 ± 9.24\n[17.07, 27.30]","28.96 ± 11.41\n[22.64, 35.28]","32.96 ± 14.69\n[24.82, 41.09]",Friedman test,χ²(2) = 16.533; p < .001; q = .001; Kendall's ...
6,RRF,15,"12.27 ± 9.25\n[7.15, 17.39]","13.57 ± 8.80\n[8.69, 18.44]","14.60 ± 8.95\n[9.64, 19.55]",Repeated-measures ANOVA,"F(2.000, 28.000) = 6.539; p = .005; q = .016; ..."
7,RTA,15,"14.01 ± 10.65\n[8.11, 19.91]","17.55 ± 11.89\n[10.96, 24.14]","20.42 ± 11.79\n[13.89, 26.95]",Repeated-measures ANOVA,"F(2.000, 28.000) = 13.607; p < .001; q < .001;..."
8,RTB,15,"31.05 ± 23.41\n[18.09, 44.02]","32.72 ± 23.03\n[19.96, 45.47]","37.12 ± 26.71\n[22.32, 51.91]",Repeated-measures ANOVA,"F(1.203, 16.836) = 4.738; GG-corrected; p = .0..."
9,RUT,15,"93.87 ± 69.78\n[55.23, 132.51]","92.87 ± 63.59\n[57.65, 128.08]","97.70 ± 58.31\n[65.41, 129.98]",Friedman test,χ²(2) = 2.800; p = .247; q = .308; Kendall's W...



Holm-adjusted post-hoc comparisons:


,Metric,Comparison,N,Test,Statistic,Raw p,Effect Size,Effect Size Type,Holm-adjusted p
0,RGAL,LOW − NR,15,Wilcoxon signed-rank,1.000000,0.000122,0.983333,Rank-biserial r,0.000366
1,RGAL,HIGH − NR,15,Wilcoxon signed-rank,9.000000,0.002014,0.850000,Rank-biserial r,0.004028
2,RGAL,HIGH − LOW,15,Wilcoxon signed-rank,24.000000,0.041260,0.600000,Rank-biserial r,0.041260
3,RRF,LOW − NR,15,Paired-samples t-test,3.066508,0.008371,0.791769,Cohen dz,0.018475
4,RRF,HIGH − NR,15,Paired-samples t-test,3.220967,0.006158,0.831650,Cohen dz,0.018475
5,RRF,HIGH − LOW,15,Paired-samples t-test,1.393356,0.185240,0.359763,Cohen dz,0.185240
6,RTA,LOW − NR,15,Paired-samples t-test,3.589936,0.002957,0.926917,Cohen dz,0.005914
7,RTA,HIGH − NR,15,Paired-samples t-test,4.594434,0.000417,1.186278,Cohen dz,0.001251
8,RTA,HIGH − LOW,15,Paired-samples t-test,2.250357,0.041026,0.581040,Cohen dz,0.041026


(  Metric  Participants                              NR  \
 0   LGAL            15   25.91 ± 14.76\n[17.73, 34.08]   
 1    LRF            15    30.75 ± 41.10\n[7.99, 53.51]   
 2    LTA            15    14.46 ± 12.00\n[7.81, 21.10]   
 3    LUT            15   68.67 ± 53.14\n[39.24, 98.10]   
 4    RBB            15   20.56 ± 12.52\n[13.63, 27.50]   
 5   RGAL            15    22.18 ± 9.24\n[17.07, 27.30]   
 6    RRF            15     12.27 ± 9.25\n[7.15, 17.39]   
 7    RTA            15    14.01 ± 10.65\n[8.11, 19.91]   
 8    RTB            15   31.05 ± 23.41\n[18.09, 44.02]   
 9    RUT            15  93.87 ± 69.78\n[55.23, 132.51]   
 
                               LOW                            HIGH  \
 0   27.76 ± 16.30\n[18.73, 36.78]   24.34 ± 13.49\n[16.87, 31.81]   
 1    27.35 ± 35.44\n[7.72, 46.98]    29.13 ± 39.57\n[7.21, 51.04]   
 2    15.12 ± 12.93\n[7.96, 22.28]    16.46 ± 13.64\n[8.91, 24.02]   
 3   68.63 ± 51.08\n[40.34, 96.92]  71.50 ± 52.94\n[42.18, 100.81]   